# Введение в JAX

**JAX** — это библиотека для быстрых численных вычислений и машинного обучения. Её идея проста:

> *JAX = NumPy + автоматическое дифференцирование + компиляция (XLA) + запуск на CPU / GPU / TPU.*

Вы пишете обычные математические функции на Python. JAX умеет:

- **дифференцировать** их (`grad`, `jvp`, `vjp`),
- **векторизовать** их (`vmap`),
- **компилировать** их в быстрый машинный код (`jit`).

Эти операции называются **преобразованиями функций** (*function transformations*). Их можно свободно комбинировать.

---

## Как работать с ноутбуком

- Ноутбук рассчитан на **Google Colab**. Всё нужное уже установлено. Для ускорения можно выбрать GPU: `Среда выполнения → Сменить тип среды выполнения`.
- Выполняйте ячейки **сверху вниз**.
- Ячейки с пометкой ✏️ **Задание** — для самостоятельной работы. После каждого задания идёт ячейка с **тестами**: запустите её, чтобы проверить решение.
  - ✅ — тест пройден, ❌ — тест не пройден (рядом есть подсказка), ⏳ — задание ещё не выполнено.
- В конце ноутбука есть раздел с эталонными решениями. Сначала попробуйте сами.
- Иногда ячейки **специально** вызывают ошибку: так мы показываем «острые углы» JAX. Такие ячейки обёрнуты в `try/except`, и ноутбук не падает.

## План

| № | Тема |
|---|------|
| 1 | Массивы и `jax.numpy` |
| 2 | Преобразования: `grad` и `vmap` |
| 3 | Pytree |
| 4 | Псевдослучайные числа |
| 5 | Состояние и функциональный стиль (пример `Counter`) |
| 6 | Компиляция `jit` |
| 7 | Компиляция заранее (AOT) |
| 8 | Управляющие конструкции и логические операторы с `jit` |
| 9 | Размещение данных (devices) |
| 10 | Автодифференцирование: JVP и VJP (с математикой) |
| 11 | JAX — The Sharp Bits (острые углы) |
| 12 | Большой пример: логистическая регрессия (softmax) → мост к нейросетям |
| 13 | Итоговые задания |
| — | Эталонные решения |

Официальная документация: <https://docs.jax.dev>

## 0. Подготовка

In [ ]:
import time
from functools import partial

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import lax

print("Версия JAX:", jax.__version__)
print("Устройства:", jax.devices())
print("Backend:", jax.default_backend())

Если в Colab у вас очень старая версия JAX, раскомментируйте и выполните ячейку ниже, затем перезапустите среду выполнения.

In [ ]:
# !pip install -U jax   # для GPU: !pip install -U "jax[cuda12]"

### Вспомогательная функция для самопроверки

Ячейка ниже определяет `run_tests(...)`. Она запускает тесты и аккуратно печатает результат. Ничего менять не нужно.

In [ ]:
def run_tests(*tests):
    ok = 0
    for t in tests:
        try:
            t()
            print(f"✅ {t.__name__}")
            ok += 1
        except AssertionError as e:
            print(f"❌ {t.__name__}: {e}")
        except NotImplementedError:
            print(f"⏳ {t.__name__}: задание ещё не выполнено")
        except Exception as e:
            print(f"❌ {t.__name__}: ошибка {type(e).__name__}: {e}")
    print(f"\nПройдено тестов: {ok} из {len(tests)}")


def show_error(fn):
    # Запускает fn и печатает ошибку (если она есть). Нужна, чтобы показывать «ожидаемые» ошибки.
    try:
        out = fn()
        print("Ошибки нет, результат:", out)
    except Exception as e:
        msg = " ".join(l.strip() for l in str(e).strip().split("\n")[:2])[:300]
        print(f"{type(e).__name__}: {msg}")

---
# 1. Массивы и `jax.numpy`

Основной тип данных JAX — массив `jax.Array`. Основной API — модуль `jax.numpy`, который почти полностью повторяет NumPy. Обычно его импортируют как `jnp`.

Что **одинаково** с NumPy: создание массивов, арифметика, срезы, редукции (`sum`, `mean`), `@` для матричного умножения, broadcasting.

Но есть важные **отличия**:

1. массивы JAX **неизменяемы** (immutable);
2. по умолчанию используется **float32**, а не float64;
3. индексация за границами массива **не бросает ошибку**;
4. функции `jnp` не принимают списки Python (нужно явно создать массив);
5. массивы **живут на устройствах** (CPU/GPU/TPU).

Рассмотрим их по очереди.

## 1.1. От NumPy к JAX

In [ ]:
x = jnp.arange(9.0).reshape(3, 3)
y = jnp.ones(3)

print(x)
print("x @ y      =", x @ y)
print("x.sum(0)   =", x.sum(axis=0))
print("x[1, :2]   =", x[1, :2])
print("тип:", type(x), "| это jax.Array?", isinstance(x, jax.Array))
print("shape:", x.shape, "dtype:", x.dtype)

JAX и NumPy хорошо дружат. Функции `jnp` принимают массивы NumPy. Типичное разделение ролей:

- `np` — загрузка данных и лёгкая работа на CPU («на хосте»);
- `jnp` — тяжёлые вычисления, которые могут идти на ускорителе.

In [ ]:
a_np = np.linspace(0, 1, 5)        # массив NumPy (float64)
a_jax = jnp.asarray(a_np)           # массив JAX (float32)
print(a_np.dtype, "->", a_jax.dtype)
print("Обратно в NumPy:", np.asarray(a_jax))

## 1.2. Массивы JAX неизменяемы

В NumPy можно менять элементы на месте (`x[0] = 10`). В JAX так делать **нельзя**. Вместо этого используется *функциональное обновление*: `x.at[idx].set(value)`. Оно возвращает **новый** массив, а старый остаётся прежним.

In [ ]:
x_np = np.arange(10)
x_np[0] = 10
print("NumPy:", x_np)

x = jnp.arange(10)
show_error(lambda: x.__setitem__(0, 10))     # ошибка: JAX arrays are immutable

In [ ]:
y = x.at[0].set(10)
print("x (не изменился):", x)
print("y (новый массив):", y)

# Другие операции .at: add, multiply, min, max, ...
print(x.at[3].add(100))
print(x.at[:3].max(5))

> **Не бойтесь копирования.** Вне `jit` каждое такое обновление действительно копирует массив. Но внутри `jax.jit` компилятор чаще всего делает обновление **на месте**. Код без мутаций проще преобразовывать, оптимизировать и распараллеливать — в этом главная причина такого дизайна.

## 1.3. Типы данных по умолчанию

NumPy по умолчанию использует 64-битные числа. JAX использует **32-битные** (быстрее и экономнее на GPU/TPU). Режим float64 по умолчанию выключен. Включить его можно флагом `jax_enable_x64` (см. раздел «Sharp Bits»).

In [ ]:
print("NumPy:", np.array([1.0, 2.0]).dtype)
print("JAX:  ", jnp.array([1.0, 2.0]).dtype)
print("JAX int:", jnp.array([1, 2]).dtype)

# Правила приведения типов похожи на NumPy, но не идентичны:
i = np.arange(3, dtype=np.int32)
h = np.ones(3, dtype=np.float16)
print("NumPy int32 + float16 ->", (i + h).dtype)
print("JAX   int32 + float16 ->", (jnp.asarray(i) + jnp.asarray(h)).dtype)

## 1.4. Индексация: отличия от NumPy

JAX-код должен компилироваться для ускорителей, а внутри вычисления нельзя «бросить исключение». Поэтому:

- **чтение** за границами массива — индекс **обрезается** (clamp) до последнего элемента;
- **запись** (`.at[...]`) за границами — обновление **молча отбрасывается**.

Поведением можно управлять аргументом `mode`.

In [ ]:
x = jnp.arange(10)
print("x[11]               =", x[11])                 # = x[9]
print("x.at[11].set(99)    =", x.at[11].set(99))      # ничего не изменилось
print("mode='fill'         =", x.at[11].get(mode="fill", fill_value=-1))

Ещё одно отличие: функции `jnp` не принимают **списки** Python. Автоматическое преобразование списков — частый источник скрытых проблем с производительностью, поэтому JAX просит сделать это явно.

In [ ]:
show_error(lambda: jnp.sum([1, 2, 3]))        # список — ошибка
print(jnp.sum(jnp.array([1, 2, 3])))           # так правильно

## 1.5. Массивы живут на устройствах

Данные массива лежат на устройстве (CPU, GPU или TPU). Атрибут `sharding` описывает, где именно. Подробнее — в разделе 9.

In [ ]:
x = jnp.arange(5)
print(x.devices())
print(x.sharding)

## 1.6. Под капотом: `jax.numpy`, `jax.lax` и XLA

`jax.numpy` — удобная оболочка. Она реализована через более строгий низкоуровневый API `jax.lax`, а тот, в свою очередь, соответствует операциям компилятора **XLA**.

`jax.lax` ничего не приводит неявно, зато содержит более общие операции (свёртки с шагами, управляющие конструкции). Если не нашли функцию в `jnp` — загляните в `lax`.

In [ ]:
print(jnp.add(1, 1.0))                       # jnp приводит типы сам
show_error(lambda: lax.add(1, 1.0))           # lax требует одинаковые типы
print(lax.add(jnp.float32(1), jnp.float32(1.0)))

## ✏️ Задание 1. Массивы и неизменяемость


Реализуйте две функции.

1. `zero_first_row(x)` — вернуть **новый** массив, в котором первая строка заменена нулями. Исходный `x` менять нельзя (используйте `.at`).
2. `standardize_columns(x)` — вернуть массив, в котором из каждого столбца вычтено среднее и результат поделён на стандартное отклонение этого столбца. **Без циклов.**

In [ ]:
def zero_first_row(x):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def standardize_columns(x):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def test_zero_first_row_values():
    x = jnp.arange(1.0, 13.0).reshape(3, 4)
    y = zero_first_row(x)
    assert isinstance(y, jax.Array), "нужно вернуть jax.Array"
    assert y.shape == x.shape, "форма должна сохраниться"
    assert bool(jnp.all(y[0] == 0)), "первая строка должна быть нулевой"
    assert bool(jnp.array_equal(y[1:], x[1:])), "остальные строки менять нельзя"


def test_zero_first_row_keeps_input():
    x = jnp.arange(1.0, 13.0).reshape(3, 4)
    _ = zero_first_row(x)
    assert float(x[0, 0]) == 1.0, "исходный массив должен остаться прежним"


def test_standardize_columns():
    x = jax.random.normal(jax.random.key(0), (100, 5)) * 3 + 7
    z = standardize_columns(x)
    assert z.shape == x.shape
    np.testing.assert_allclose(z.mean(axis=0), 0.0, atol=1e-4, err_msg="среднее каждого столбца должно быть 0")
    np.testing.assert_allclose(z.std(axis=0), 1.0, atol=1e-4, err_msg="std каждого столбца должно быть 1")


run_tests(test_zero_first_row_values, test_zero_first_row_keeps_input, test_standardize_columns)

---
# 2. Преобразования: `grad` и `vmap`

Сердце JAX — **преобразования функций**. Это функции высшего порядка: они берут вашу функцию и возвращают новую, которая считает что-то связанное с ней.

| Преобразование | Что делает |
|---|---|
| `jax.grad(f)` | возвращает функцию, считающую **градиент** $\nabla f$ |
| `jax.vmap(f)` | возвращает **векторизованную** версию $f$ (работает по батчу) |
| `jax.jit(f)` | возвращает **скомпилированную** версию $f$ (раздел 6) |

Преобразования **комбинируются**: можно дифференцировать векторизованную функцию, векторизовать производную, скомпилировать всё вместе.

## 2.1. Автоматическое дифференцирование: `jax.grad`

`jax.grad(f)` принимает функцию, возвращающую **скаляр**, и возвращает функцию, считающую градиент.

In [ ]:
grad_tanh = jax.grad(jnp.tanh)
print(grad_tanh(2.0))                       # 1 - tanh(2)^2

# Проверка: 1 - tanh^2(2)
print(1 - jnp.tanh(2.0) ** 2)

Так как `grad` переводит функцию в функцию, его можно применять **повторно**, получая производные высших порядков.

In [ ]:
f = lambda x: x**3 + 2 * x**2 - 3 * x + 1

dfdx = jax.grad(f)
d2fdx = jax.grad(dfdx)
d3fdx = jax.grad(d2fdx)

print(dfdx(1.0))     # 3x^2 + 4x - 3  при x=1 -> 4
print(d2fdx(1.0))    # 6x + 4         при x=1 -> 10
print(d3fdx(1.0))    # 6

### Дифференцирование по разным аргументам

По умолчанию `grad` берёт производную по **первому** аргументу. Параметр `argnums` выбирает другие аргументы (или сразу несколько). Пример — функция потерь логистической регрессии:

In [ ]:
def sigmoid(x):
    return 0.5 * (jnp.tanh(x / 2) + 1)

def predict(W, b, inputs):
    return sigmoid(jnp.dot(inputs, W) + b)

inputs = jnp.array([[0.52, 1.12, 0.77],
                    [0.88, -1.08, 0.15],
                    [0.52, 0.06, -1.30],
                    [0.74, -2.49, 1.39]])
targets = jnp.array([True, True, False, True])
W = jnp.array([0.1, 0.4, -0.3])
b = 0.5

def loss(W, b):
    preds = predict(W, b, inputs)
    label_probs = preds * targets + (1 - preds) * (1 - targets)
    return -jnp.sum(jnp.log(label_probs))

W_grad = jax.grad(loss, argnums=0)(W, b)
b_grad = jax.grad(loss, argnums=1)(W, b)
W_grad2, b_grad2 = jax.grad(loss, argnums=(0, 1))(W, b)
print("dL/dW =", W_grad)
print("dL/db =", b_grad)

### `value_and_grad` и дополнительные выходы

Обычно нужен и градиент, и само значение функции (например, чтобы печатать loss). `jax.value_and_grad` считает и то и другое за один проход. Если функция возвращает ещё и вспомогательные данные, возвращайте пару `(скаляр, aux)` и передайте `has_aux=True`.

In [ ]:
loss_value, (gW, gb) = jax.value_and_grad(loss, (0, 1))(W, b)
print("loss =", loss_value)

def loss_and_preds(W, b):
    preds = predict(W, b, inputs)
    label_probs = preds * targets + (1 - preds) * (1 - targets)
    return -jnp.sum(jnp.log(label_probs)), preds

gW, preds = jax.grad(loss_and_preds, has_aux=True)(W, b)
print("preds =", preds)

### Проверка производных численно

Производные легко проверить конечными разностями. В JAX есть готовая функция `check_grads`.

In [ ]:
from jax.test_util import check_grads

eps = 1e-3
numeric = (loss(W, b + eps / 2) - loss(W, b - eps / 2)) / eps
print("численно:", numeric, "| автоматически:", jax.grad(loss, 1)(W, b))

check_grads(loss, (W, b), order=2)      # проверяет производные до 2-го порядка
print("check_grads пройден")

## 2.2. Автоматическая векторизация: `jax.vmap`

`jax.vmap` превращает функцию, написанную для **одного** примера, в функцию для **батча** примеров. Никаких циклов Python и ручной перестройки кода.

Пример: скалярное произведение окна на веса («свёртка» 1D).

In [ ]:
x = jnp.arange(5.0)
w = jnp.array([2.0, 3.0, 4.0])

def convolve(x, w):
    output = []
    for i in range(1, len(x) - 1):
        output.append(jnp.dot(x[i - 1:i + 2], w))
    return jnp.array(output)

print(convolve(x, w))

xs = jnp.stack([x, x])
ws = jnp.stack([w, w])

auto_batch_convolve = jax.vmap(convolve)
print(auto_batch_convolve(xs, ws))

Хотя функция написана для одного примера, после `vmap` каждая операция внутри работает **сразу со всем батчем**. Посмотрим на это глазами: `jax.make_jaxpr` показывает программу, которую JAX записал при трассировке.

In [ ]:
print(jax.make_jaxpr(convolve)(x, w))
print()
print(jax.make_jaxpr(jax.vmap(convolve))(xs, ws))

Видно, что программа та же, но у каждой операции появилась **ось батча** (`f32[5]` → `f32[2,5]`). Размер программы не растёт с размером батча.

### `in_axes` и `out_axes`

По умолчанию `vmap` идёт по **первой** оси каждого аргумента. Это настраивается:

- `in_axes=1` — батч по второй оси;
- `in_axes=(0, None)` — первый аргумент по батчу, второй **общий** для всех (не размножается).

In [ ]:
# общий вектор весов w для всех x
batch_conv_shared_w = jax.vmap(convolve, in_axes=(0, None))
print(batch_conv_shared_w(xs, w))

# батч по второй оси
conv_v2 = jax.vmap(convolve, in_axes=1, out_axes=1)
print(conv_v2(xs.T, ws.T))

### Композиция: «все пары» и градиенты по примерам

`vmap` можно вкладывать. Это даёт короткую запись для вычислений «все пары»:

In [ ]:
def dist(x, y):
    return jnp.sqrt(jnp.sum((x - y) ** 2))

def all_pairs(f):
    return jax.vmap(jax.vmap(f, in_axes=(None, 0)), in_axes=(0, None))

points = jnp.array([[0., 0.], [1., 0.], [0., 2.]])
print(all_pairs(dist)(points, points))

Комбинация `vmap` и `grad` решает задачи, которые иначе записать неудобно. Например, **градиент потерь для каждого примера отдельно** (а не суммарный по батчу). Это нужно, например, для дифференциальной приватности или для анализа вклада примеров.

In [ ]:
def example_loss(W, b, x, y):
    pred = predict(W, b, x)
    label_prob = pred * y + (1 - pred) * (1 - y)
    return -jnp.log(label_prob)

per_example_grads = jax.vmap(jax.grad(example_loss, (0, 1)), in_axes=(None, None, 0, 0))
W_grads, b_grads = per_example_grads(W, b, inputs, targets)
print("Градиенты по W (одна строка — один пример):\n", W_grads)
print("Градиенты по b:", b_grads)

## 2.3. Как работают преобразования: трассировка

Чтобы преобразовать функцию, JAX должен знать, **что она делает**. Он узнаёт это через **трассировку** (*tracing*): вызывает вашу функцию Python, подставляя вместо массивов специальные объекты — **трейсеры** (*tracers*). Трейсер записывает каждую операцию, применённую к нему.

In [ ]:
def f(x):
    print("x =", x)                      # print сработает на этапе трассировки!
    return x * 2

result = jax.vmap(f)(jnp.arange(3))
print("результат:", result)

Мы увидели **не массив**, а трейсер. Заметьте: батч состоит из 3 элементов, но `print` сработал **один раз**. Функция выполняется на трассировке один раз, а не по разу на элемент.

### Следствие 1: функции должны быть «чистыми»

Преобразования надёжно работают для **чистых** (pure) функций:

- результат зависит только от аргументов;
- результат получается применением операций JAX;
- **нет побочных эффектов** (print, запись в глобальные переменные, изменение объектов).

Побочные эффекты (например, `print`) происходят на этапе трассировки, а не при каждом запуске. Для печати значений во время работы используйте `jax.debug.print`.

### Следствие 2: трейсер не знает значения

Трассированный код не может ветвиться по значению данных: нужно выбирать одну ветку, а каждый элемент батча может хотеть свою.

In [ ]:
def absolute_bad(x):
    if x > 0:              # трейсер не знает, больше ли он нуля
        return x
    else:
        return -x

show_error(lambda: jax.vmap(absolute_bad)(jnp.arange(-2.0, 3.0)))

In [ ]:
def absolute(x):
    return jnp.where(x > 0, x, -x)      # поэлементный выбор

print(jax.vmap(absolute)(jnp.arange(-2.0, 3.0)))

### Особый случай: `grad` без `jit` работает с обычным Python-кодом

Когда `grad` применяется **сам по себе**, трейсер несёт **одно** конкретное значение. Поэтому `if`, `while` и рекурсия работают как обычно.

In [ ]:
def f(x):
    if x < 3:
        return 3.0 * x ** 2
    else:
        return -4.0 * x

print(jax.grad(f)(2.0))   # ветка 3x^2: 6x = 12
print(jax.grad(f)(4.0))   # ветка -4x: -4

def sqrt_newton(a):
    x = a
    while abs(x * x - a) > 1e-6:     # количество итераций зависит от данных
        x = 0.5 * (x + a / x)
    return x

print(sqrt_newton(2.0), jax.grad(sqrt_newton)(2.0), "(ожидаем 1/(2*sqrt(2)) ≈ 0.3536)")

Но стоит добавить `jit` или `vmap` поверх такой функции — ограничения возвращаются. Для таких случаев есть `jnp.where`, `lax.cond`, `lax.scan` (раздел 8).

## ✏️ Задание 2. grad и vmap


Дана функция $f_2(x)=\sum_i x_i^3+\sin(x_0)$.

1. `grad_f2(x)` — вернуть градиент $f_2$ в точке `x` (используйте `jax.grad`).
2. `batch_grad_f2(xs)` — для массива `xs` формы `(N, d)` вернуть массив формы `(N, d)`, где i-я строка — градиент $f_2$ в точке `xs[i]`. Используйте `jax.vmap`, **без циклов**.

In [ ]:
def f2(x):
    return jnp.sum(x ** 3) + jnp.sin(x[0])


def grad_f2(x):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def batch_grad_f2(xs):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def test_grad_f2():
    x = jnp.array([1.0, 2.0, 3.0])
    expected = 3 * x ** 2 + jnp.array([jnp.cos(1.0), 0.0, 0.0])
    np.testing.assert_allclose(grad_f2(x), expected, rtol=1e-5, err_msg="градиент посчитан неверно")


def test_batch_grad_f2_shape_and_values():
    xs = jax.random.normal(jax.random.key(1), (6, 4))
    g = batch_grad_f2(xs)
    assert g.shape == xs.shape, f"ожидалась форма {xs.shape}, получена {g.shape}"
    for i in range(xs.shape[0]):
        np.testing.assert_allclose(g[i], grad_f2(xs[i]), rtol=1e-5, atol=1e-6,
                                   err_msg=f"строка {i} не совпадает с градиентом для одного примера")


run_tests(test_grad_f2, test_batch_grad_f2_shape_and_values)

---
# 3. Pytree

Функции JAX работают с массивами, но настоящие программы передают **структуры**: словарь параметров нейросети, список батчей, вложенные конфигурации. JAX называет такие вложенные структуры **pytree**.

## 3.1. Что такое pytree

**Pytree** — рекурсивная структура: это либо **лист** (leaf), либо **контейнер** (node), чьи дети тоже pytree. Контейнеры по умолчанию: `list`, `tuple`, `dict` (и родственные: `namedtuple`, `OrderedDict`). Всё остальное (массив, число, строка…) — лист.

In [ ]:
example_trees = [
    [1, "a", object()],
    (1, (2, 3), ()),
    [1, {"k1": 2, "k2": (3, 4)}, 5],
    {"a": 2, "b": (2, 3)},
    jnp.array([1, 2, 3]),
]

for pytree in example_trees:
    leaves = jax.tree.leaves(pytree)
    print(f"{repr(pytree):<45} -> {len(leaves)} листьев")

Любой pytree делится на две части: **листья** (данные) и **treedef** (структура). `jax.tree.flatten` делит, `jax.tree.unflatten` собирает обратно. Именно так преобразования JAX поддерживают pytree: внутри они работают с плоским списком массивов, а потом восстанавливают вашу структуру.

In [ ]:
params = {"W": jnp.zeros((2, 3)), "b": jnp.zeros(3)}

leaves, treedef = jax.tree.flatten(params)
print("листья:", leaves)
print("структура:", treedef)
print("собрали обратно:", jax.tree.unflatten(treedef, leaves))

## 3.2. Основные функции

Чаще всего нужен `jax.tree.map` — аналог `map`, который идёт по всем листьям. Он умеет работать с **несколькими** pytree одинаковой структуры сразу. Ещё полезны `jax.tree.reduce` (свёртка по листьям) и `jax.tree.structure`.

In [ ]:
list_of_lists = [[1, 2, 3], [1, 2], [1, 2, 3, 4]]

print(jax.tree.map(lambda x: x * 2, list_of_lists))
print(jax.tree.map(lambda x, y: x + y, list_of_lists, list_of_lists))
print("сумма всех листьев:", jax.tree.reduce(lambda a, b: a + b, list_of_lists))

## 3.3. Pytree и преобразования

Все преобразования JAX принимают и возвращают pytree. Градиент по словарю параметров вернётся **словарём той же структуры**. Это и делает JAX удобным для машинного обучения:

> параметры → pytree; `jax.grad` → pytree градиентов; `jax.tree.map` → шаг оптимизации.

In [ ]:
def loss(params, x):
    pred = jnp.dot(x, params["W"]) + params["b"]
    return jnp.sum(pred ** 2)

params = {"W": jnp.ones(3), "b": 0.5}
x = jnp.array([[1., 2., 3.], [4., 5., 6.]])

print(jax.grad(loss)(params, x))

Полный пример: обучим маленькую нейросеть (MLP) на $y = x^2$. Параметры — **список словарей**.

In [ ]:
def init_mlp_params(layer_widths, seed=0):
    rng = np.random.default_rng(seed)
    params = []
    for n_in, n_out in zip(layer_widths[:-1], layer_widths[1:]):
        params.append(dict(
            weights=jnp.asarray(rng.normal(size=(n_in, n_out)) * np.sqrt(2 / n_in), dtype=jnp.float32),
            biases=jnp.ones((n_out,), dtype=jnp.float32),
        ))
    return params

params = init_mlp_params([1, 128, 128, 1])
print(jax.tree.map(lambda x: x.shape, params))

In [ ]:
def forward(params, x):
    *hidden, last = params
    for layer in hidden:
        x = jax.nn.relu(x @ layer["weights"] + layer["biases"])
    return x @ last["weights"] + last["biases"]

def loss_fn(params, x, y):
    return jnp.mean((forward(params, x) - y) ** 2)

LEARNING_RATE = 1e-3

@jax.jit
def update(params, x, y):
    grads = jax.grad(loss_fn)(params, x, y)               # pytree той же структуры, что params
    return jax.tree.map(lambda p, g: p - LEARNING_RATE * g, params, grads)

x = jnp.asarray(np.random.default_rng(1).normal(size=(128, 1)), dtype=jnp.float32)
y = x ** 2

print("loss до:", loss_fn(params, x, y))
for _ in range(300):
    params = update(params, x, y)
print("loss после:", loss_fn(params, x, y))

> Обратите внимание: шаг SGD — это **одна строка** `jax.tree.map`. Она одинаково работает для логистической регрессии и для нейросети с любым числом слоёв.

## 3.4. Пути к листьям (key paths)

У каждого листа есть «путь» от корня. Он полезен для отладки и логирования по именам параметров.

In [ ]:
import collections
ATuple = collections.namedtuple("ATuple", ("name",))

tree = [1, {"k1": 2, "k2": (3, 4)}, ATuple("foo")]
flattened, _ = jax.tree_util.tree_flatten_with_path(tree)

for key_path, value in flattened:
    print(f"tree{jax.tree_util.keystr(key_path)} = {value}")

## 3.5. Свои классы как pytree

Свой класс по умолчанию — **лист**, даже если внутри лежат массивы. Тогда `tree.map` не заходит внутрь:

In [ ]:
class Special:
    def __init__(self, name, x, y):
        self.name, self.x, self.y = name, x, y

print(jax.tree.leaves([Special("a", 0, 1), Special("b", 2, 4)]))   # два объекта-листа
show_error(lambda: jax.tree.map(lambda x: x + 1, [Special("a", 0, 1), Special("b", 2, 4)]))

Чтобы класс стал контейнером, его **регистрируют**. Нужно сказать, как:

- **разобрать** объект на `children` (динамические данные, массивы) и `aux_data` (статические метаданные);
- **собрать** объект обратно.

`aux_data` попадает в `treedef`, поэтому должна поддерживать сравнение и хеширование.

In [ ]:
from jax.tree_util import register_pytree_node_class

@register_pytree_node_class
class RegSpecial(Special):
    def __repr__(self):
        return f"RegSpecial({self.name!r}, x={self.x}, y={self.y})"

    def tree_flatten(self):
        children = (self.x, self.y)     # динамические данные
        aux_data = self.name            # статические метаданные
        return children, aux_data

    @classmethod
    def tree_unflatten(cls, aux_data, children):
        return cls(aux_data, *children)

print(jax.tree.map(lambda x: x + 1, [RegSpecial("a", 0, 1), RegSpecial("b", 2, 4)]))
print(jax.grad(lambda s: s.x ** 2 + s.y)(RegSpecial("a", 3.0, 4.0)))   # градиент — тоже RegSpecial

Для `dataclass` есть короткий путь — `jax.tree_util.register_dataclass`. Он позволяет явно указать, какие поля — данные, а какие — метаданные.

In [ ]:
from dataclasses import dataclass
from typing import Any

@partial(jax.tree_util.register_dataclass, data_fields=["a", "b"], meta_fields=["name"])
@dataclass
class Container:
    name: str
    a: Any
    b: Any

print(jax.tree.leaves([Container("apple", 5.3, 1.2), Container("banana", jnp.zeros(4), -1.0)]))

@jax.jit
def g(c):
    return c.a + c.b

print(g(Container("mdc", 1.0, 2.0)))   # name — статическое поле, jit не жалуется

## 3.6. Типичные ловушки

**1. Узел принят за лист.** `.shape` — это кортеж, т.е. узел, а не лист.

In [ ]:
a_tree = [jnp.zeros((2, 3)), jnp.zeros((3, 4))]
shapes = jax.tree.map(lambda x: x.shape, a_tree)
print(shapes)
# jnp.ones вызовется для 2 и 3 по отдельности, а не для (2, 3):
print(jax.tree.map(jnp.ones, shapes))
# исправление: is_leaf говорит, что кортеж — лист
print(jax.tree.map(jnp.ones, shapes, is_leaf=lambda x: isinstance(x, tuple)))

**2. `None` — пустой узел, а не лист.** **3. Ключи словаря сортируются** при разборе, поэтому смешивать ключи несравнимых типов (`int` и `str`) нельзя.

In [ ]:
print(jax.tree.leaves([None, None, None]))                                  # []
print(jax.tree.leaves([None, None, None], is_leaf=lambda x: x is None))     # [None, None, None]
show_error(lambda: jax.tree.map(lambda x: x + 1, {1: 7, "y": 42}))

## 3.7. Полезный приём: «список деревьев» → «дерево списков»

In [ ]:
def tree_transpose(list_of_trees):
    return jax.tree.map(lambda *xs: list(xs), *list_of_trees)

episode_steps = [dict(t=1, obs=3), dict(t=2, obs=4)]
print(tree_transpose(episode_steps))

## ✏️ Задание 3. Pytree


Реализуйте две функции для **произвольных** pytree из массивов.

1. `tree_scale(tree, c)` — вернуть pytree той же структуры, в котором каждый лист умножен на `c`.
2. `global_norm(tree)` — вернуть евклидову норму «всех параметров сразу»: $\sqrt{\sum_{\text{листья}} \sum_i x_i^2}$. Это используется, например, при *gradient clipping*.

Подсказка: `jax.tree.map`, `jax.tree.leaves`.

In [ ]:
def tree_scale(tree, c):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def global_norm(tree):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def _example_tree():
    return {"a": jnp.array([3.0, 4.0]), "b": [jnp.array(12.0), {"c": jnp.zeros(2)}]}


def test_tree_scale_structure():
    t = _example_tree()
    s = tree_scale(t, 2.0)
    assert jax.tree.structure(s) == jax.tree.structure(t), "структура pytree должна сохраниться"


def test_tree_scale_values():
    s = tree_scale(_example_tree(), 2.0)
    np.testing.assert_allclose(s["a"], [6.0, 8.0])
    np.testing.assert_allclose(s["b"][0], 24.0)


def test_global_norm():
    n = global_norm(_example_tree())
    np.testing.assert_allclose(n, 13.0, rtol=1e-5, err_msg="sqrt(9+16+144) = 13")


def test_global_norm_is_jittable():
    n = jax.jit(global_norm)(_example_tree())
    np.testing.assert_allclose(n, 13.0, rtol=1e-5)


run_tests(test_tree_scale_structure, test_tree_scale_values, test_global_norm, test_global_norm_is_jittable)

---
# 4. Псевдослучайные числа (кратко)

В NumPy генератор хранит **скрытое глобальное состояние**: каждый вызов `np.random.normal()` его меняет. Это побочный эффект, который не дружит с трассировкой и параллелизмом.

В JAX всё **явно**. Генератор — это **ключ** (`key`). Правила:

1. Ключ создаётся из числа: `jax.random.key(seed)`.
2. Одинаковый ключ → одинаковые числа. Ключ **нельзя использовать дважды**: иначе вы получите одни и те же числа.
3. Чтобы получить новые ключи, ключ **расщепляют**: `key1, key2 = jax.random.split(key)`.

In [ ]:
key = jax.random.key(42)

print("Один и тот же ключ дважды:")
print(jax.random.normal(key, (3,)))
print(jax.random.normal(key, (3,)))      # те же числа!

print("\nПравильно: расщепляем ключ")
key, subkey1, subkey2 = jax.random.split(key, 3)
print(jax.random.normal(subkey1, (3,)))
print(jax.random.normal(subkey2, (3,)))

Типичный паттерн в цикле: на каждой итерации делаем `key, sub = jax.random.split(key)` и используем `sub`. Другие полезные функции: `uniform`, `bernoulli`, `randint`, `permutation`, `choice`, `fold_in` (получить ключ из ключа и числа, например номера шага).

In [ ]:
key = jax.random.key(0)
key, k1, k2, k3 = jax.random.split(key, 4)

print("uniform    :", jax.random.uniform(k1, (3,)))
print("bernoulli  :", jax.random.bernoulli(k2, 0.3, (6,)))
print("permutation:", jax.random.permutation(k3, 8))
print("fold_in    :", jax.random.normal(jax.random.fold_in(key, 7), (2,)))

# vmap по ключам: независимые случайные величины
keys = jax.random.split(key, 4)
print(jax.vmap(lambda k: jax.random.normal(k, ()))(keys))

**Зачем так сложно?** Детерминированность и воспроизводимость при `jit`, `vmap` и на многих устройствах: результат не зависит от порядка выполнения операций.

## ✏️ Задание 4. Случайные числа


Реализуйте `two_noises(key, n)`: вернуть пару `(a, b)` из **двух независимых** массивов формы `(n,)` со стандартным нормальным распределением. Из одного `key` нужно получить два разных подключа.

Типичная ошибка — использовать один и тот же ключ для обоих массивов. Тогда `a` и `b` совпадут.

In [ ]:
def two_noises(key, n):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def test_shapes():
    a, b = two_noises(jax.random.key(0), 5)
    assert a.shape == (5,) and b.shape == (5,), "оба массива должны иметь форму (n,)"


def test_different_arrays():
    a, b = two_noises(jax.random.key(0), 100)
    assert not bool(jnp.allclose(a, b)), "a и b не должны совпадать (вы использовали один ключ дважды?)"


def test_reproducible():
    a1, b1 = two_noises(jax.random.key(3), 10)
    a2, b2 = two_noises(jax.random.key(3), 10)
    assert bool(jnp.array_equal(a1, a2)) and bool(jnp.array_equal(b1, b2)), "при том же ключе результат должен быть тем же"


def test_independent():
    a, b = two_noises(jax.random.key(1), 5000)
    corr = float(jnp.corrcoef(a, b)[0, 1])
    assert abs(corr) < 0.1, f"корреляция слишком велика: {corr:.3f}"


run_tests(test_shapes, test_different_arrays, test_reproducible, test_independent)

---
# 5. Состояние и функциональный стиль

Реальным программам нужно состояние: параметры модели, моменты оптимизатора, счётчики, статистики. Но трассировка требует **чистых функций** (раздел 2.3). Как совместить?

Есть два способа:

1. **Передавать состояние через аргументы и возвращаемое значение** (классический способ, используется во всей экосистеме JAX).
2. **Refs** — явные изменяемые массивы (более новый механизм).

## 5.1. Проблема: пример `Counter`

Обычный счётчик на Python:

In [ ]:
class Counter:
    # Простой счётчик: состояние n меняется как побочный эффект.

    def __init__(self):
        self.n = 0

    def count(self) -> int:
        self.n += 1
        return self.n

counter = Counter()
for _ in range(3):
    print(counter.count())

В обычном Python всё работает. Но под JAX:

In [ ]:
counter = Counter()
print(jax.make_jaxpr(counter.count)())    # что записал JAX?

fast_count = jax.jit(Counter().count)
print([int(fast_count()) for _ in range(3)])     # всегда 1!

Записанная программа **не принимает входов, ничего не делает и всегда возвращает 1**. Побочный эффект `self.n += 1` выполнился один раз при трассировке, а значение было «зашито» как константа. Под `jit` результат трассировки кешируется, поэтому счётчик навсегда застрянет на единице.

## 5.2. Решение: состояние как аргумент и результат

Превращаем класс вида

```python
class StatefulClass:
    state: State
    def stateful_method(*args) -> Output: ...
```

в чистую функцию вида

```python
def stateless_method(state: State, *args) -> tuple[State, Output]: ...
```

In [ ]:
CounterState = int

class CounterV2:
    def count(self, n: CounterState):
        # возвращаем новое состояние и значение
        return n + 1, n + 1

    def reset(self) -> CounterState:
        return 0

counter = CounterV2()
state = counter.reset()
for _ in range(3):
    state, value = counter.count(state)
    print(value)

print(jax.make_jaxpr(counter.count)(0))       # теперь состояние видно в программе: вход -> выход

In [ ]:
fast_count = jax.jit(counter.count)
state = 0
for _ in range(3):
    state, value = fast_count(state)
    print(int(value), end=" ")

Теперь всё работает под `jit`, `vmap` и `grad`. Цена — состояние нужно явно «протаскивать» (*threading*) через код. Выигрыш — прозрачность и композиция преобразований: можно дифференцировать шаг обновления или запустить `vmap` по независимым обучающим запускам.

## 5.3. Состояние в цикле обучения

Цикл обучения — главный пример. Параметры и состояние оптимизатора (импульс) — это pytree, которые проходят через чистую функцию `update`.

In [ ]:
def loss_fn_lin(params, batch):
    x, y = batch
    return jnp.mean((params["w"] * x + params["b"] - y) ** 2)

def optimizer_step(params, grads, opt_state, lr=0.1, decay=0.9):
    # градиентный спуск с импульсом: импульс — состояние оптимизатора
    new_opt_state = jax.tree.map(lambda m, g: decay * m + g, opt_state, grads)
    new_params = jax.tree.map(lambda p, m: p - lr * m, params, new_opt_state)
    return new_params, new_opt_state

@jax.jit
def update(params, opt_state, batch):
    grads = jax.grad(loss_fn_lin)(params, batch)
    return optimizer_step(params, grads, opt_state)

params = {"w": jnp.float32(1.0), "b": jnp.float32(0.0)}
opt_state = jax.tree.map(jnp.zeros_like, params)
batch = (jnp.array([1.0, 2.0, 3.0]), jnp.array([3.0, 5.0, 7.0]))      # y = 2x + 1

for step in range(100):
    params, opt_state = update(params, opt_state, batch)

print(jax.tree.map(lambda x: round(float(x), 2), params))

Такой же контракт `update(grads, opt_state) -> (updates, new_opt_state)` используют библиотеки оптимизаторов (например, **Optax**), а библиотеки нейросетей хранят параметры как pytree.

## 5.4. Цикл с состоянием внутри `jit`: `lax.scan`

Если цикл нужно выполнить **внутри** компилируемой функции, используют `jax.lax.scan`. Он принимает функцию вида `(carry, x) -> (new_carry, y)`. `carry` — это и есть состояние.

In [ ]:
def step(carry, x):
    new_carry = carry + x          # накопленная сумма — состояние
    return new_carry, new_carry    # (новое состояние, что записать в выход)

final, history = lax.scan(step, 0.0, jnp.arange(1.0, 6.0))
print("итог:", final)
print("история:", history)

## 5.5. Refs: изменяемые массивы (кратко)

Иногда передавать состояние через все функции неудобно. Для этого в JAX есть **refs** — изменяемые массивы, которые можно читать и менять **на месте**, в том числе внутри `jit`:

```python
x_ref = jax.new_ref(jnp.zeros(3))

@jax.jit
def f():
    x_ref[1] += 1.0

f(); f()
print(x_ref)          # Ref([0., 2., 0.])
```

Читать значение ref целиком: `x_ref[...]`. Закончив, `jax.freeze(x_ref)` превращает его в обычный массив. У refs есть ограничения (нельзя вернуть ref из `jit`, нельзя передать один ref дважды и т.д.). Основной стиль в экосистеме по-прежнему — передача состояния как значений.

In [ ]:
if hasattr(jax, "new_ref"):
    x_ref = jax.new_ref(jnp.zeros(3))

    @jax.jit
    def f():
        x_ref[1] += 1.0

    f(); f()
    print(x_ref[...])
else:
    print("В этой версии JAX нет jax.new_ref (появился в новых версиях) — пропускаем.")

## ✏️ Задание 5. Чистая функция с состоянием


Реализуйте **скользящее среднее** в функциональном стиле.

Состояние — пара `(count, mean)`.

1. `update_mean(state, x)` — принять состояние и новое число `x`, вернуть пару `(new_state, new_mean)`, где $\text{mean}_{new}=\text{mean}+\frac{x-\text{mean}}{count+1}$, а `new_state = (count + 1, new_mean)`. Функция должна быть **чистой**.
2. `running_means(xs)` — с помощью `jax.lax.scan` вернуть массив всех промежуточных средних для массива `xs`. Начальное состояние: `(jnp.float32(0), jnp.float32(0))`.

In [ ]:
def update_mean(state, x):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def running_means(xs):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def test_update_mean_python_loop():
    state = (0.0, 0.0)
    got = []
    for x in [1.0, 2.0, 3.0, 4.0]:
        state, m = update_mean(state, x)
        got.append(float(m))
    np.testing.assert_allclose(got, [1.0, 1.5, 2.0, 2.5], rtol=1e-6)
    assert float(state[0]) == 4.0, "первый элемент состояния — счётчик"


def test_update_mean_is_pure():
    s0 = (jnp.float32(2.0), jnp.float32(5.0))
    r1 = update_mean(s0, 7.0)
    r2 = update_mean(s0, 7.0)
    np.testing.assert_allclose(float(r1[1]), float(r2[1]))
    assert float(s0[0]) == 2.0 and float(s0[1]) == 5.0, "входное состояние менять нельзя"


def test_update_mean_jit():
    s0 = (jnp.float32(0), jnp.float32(0))
    _, m = jax.jit(update_mean)(s0, jnp.float32(5.0))
    np.testing.assert_allclose(float(m), 5.0)


def test_running_means_scan():
    xs = jax.random.normal(jax.random.key(0), (50,))
    expected = jnp.cumsum(xs) / jnp.arange(1, 51)
    np.testing.assert_allclose(running_means(xs), expected, rtol=1e-4, atol=1e-5)


run_tests(test_update_mean_python_loop, test_update_mean_is_pure, test_update_mean_jit, test_running_means_scan)

---
# 6. Компиляция: `jax.jit`

`jax.jit` — преобразование, которое делает код JAX **быстрым**. Оно компилирует функцию Python в один оптимизированный вычислительный блок под ваше устройство (CPU, GPU или TPU).

Без `jit` JAX выполняет операции **по одной**. Это гибко, но медленно по двум причинам:

1. у каждого вызова есть накладные расходы Python и runtime;
2. компилятор не видит вычисление целиком и не может оптимизировать **между** операциями (слияние операций, исключение временных массивов).

`jit` трассирует функцию в jaxpr (раздел 2.3) и передаёт её компилятору **XLA**. Результат кешируется.

## 6.1. Что даёт компиляция

Для измерения времени пишем маленький помощник. Обратите внимание на `block_until_ready()`: JAX работает **асинхронно** (см. 6.7), и без него мы измерим только запуск вычисления.

In [ ]:
def bench(f, *args, n=30):
    # среднее время одного вызова в миллисекундах (после разогрева)
    jax.block_until_ready(f(*args))          # разогрев: трассировка + компиляция
    t0 = time.perf_counter()
    for _ in range(n):
        jax.block_until_ready(f(*args))
    return (time.perf_counter() - t0) / n * 1e3

In [ ]:
def selu(x, alpha=1.67, lambda_=1.05):
    return lambda_ * jnp.where(x > 0, x, alpha * jnp.exp(x) - alpha)

x = jnp.arange(1_000_000, dtype=jnp.float32)

selu_jit = jax.jit(selu)

print(f"без jit: {bench(selu, x):.3f} мс")
print(f"с jit:   {bench(selu_jit, x):.3f} мс")

Первый вызов `selu_jit` выполняет **трассировку и компиляцию**. Последующие вызовы с теми же типами входов сразу идут в скомпилированный код, и Python-функция `selu` больше не выполняется. Поэтому «разогрев» делают вне замера.

`jit` чаще всего применяют как декоратор:

In [ ]:
@jax.jit
def sum_sq(x):
    return jnp.sum(x ** 2)

print(sum_sq(jnp.arange(3.0)))

## 6.2. Что запоминает скомпилированная функция

Python-ветвления и побочные эффекты выполняются на этапе трассировки. Значит, в скомпилированном коде остаётся **только та ветка, что была выбрана при трассировке**, и никаких `print`.

In [ ]:
@jax.jit
def noisy(x):
    print("  [трассировка] x =", x)             # выполнится только при трассировке
    jax.debug.print("  [выполнение] x = {}", x)  # выполняется при каждом запуске
    return x * 2

print("вызов 1:"); noisy(jnp.arange(3.0))
print("вызов 2 (тот же тип):"); noisy(jnp.arange(3.0))
print("вызов 3 (другая форма -> новая трассировка):"); noisy(jnp.arange(4.0))

In [ ]:
def log2_if_rank_2(x):
    if x.ndim == 2:
        return jnp.log(x) / jnp.log(2.0)
    else:
        return x

print(jax.make_jaxpr(log2_if_rank_2)(jnp.array([1, 2, 3])))              # ранг 1: тождественная функция
print(jax.make_jaxpr(log2_if_rank_2)(jnp.ones((2, 2))))                   # ранг 2: другая ветка

Чтение **глобальной** переменной тоже опасно: её значение «зашивается» при трассировке.

In [ ]:
scale = 2.0

@jax.jit
def times_scale(x):
    return x * scale

print(times_scale(3.0))      # 6.0
scale = 10.0
print(times_scale(3.0))      # всё ещё 6.0! Старое значение зашито в скомпилированный код

## 6.3. Почему нельзя просто «заджитить всё»

Трейсеры `jit` **не несут значений**, только тип (форма и dtype). Поэтому ветвление по значению не работает.

In [ ]:
def f(x):
    if x > 0:
        return x
    else:
        return 2 * x

show_error(lambda: jax.jit(f)(10))

In [ ]:
def g(x, n):
    i = 0
    while i < n:
        i += 1
    return x + i

show_error(lambda: jax.jit(g)(10, 20))

Что делать:

- выразить условие через `jnp.where`, `lax.cond`, `lax.while_loop`, `lax.scan` (раздел 8);
- сделать аргумент **статическим**;
- заджитить только «тяжёлую» часть, а цикл оставить на Python.

## 6.4. Статические аргументы

Параметры `static_argnums` / `static_argnames` говорят `jit`: «этот аргумент — обычное значение Python, фиксированное на этапе трассировки». Для каждого нового значения будет **новая компиляция**. Значения должны быть **хешируемыми**.

In [ ]:
f_jit = jax.jit(f, static_argnums=0)
print(f_jit(10))

g_jit = jax.jit(g, static_argnames=["n"])
print(g_jit(10, 20))

# как декоратор (через partial):
@partial(jax.jit, static_argnames=["n"])
def g_decorated(x, n):
    i = 0
    while i < n:
        i += 1
    return x + i

print(g_decorated(10, 20))

## 6.5. `jit` и кеш

Ключ кеша состоит из:

- **типов** аргументов (новая форма или dtype → новая компиляция);
- значений **статических** аргументов;
- **идентичности** (object id) самой функции.

Из последнего пункта следует правило: **не вызывайте `jax.jit` на временных функциях внутри циклов**. Новая `lambda` или `partial` на каждой итерации — это каждый раз новая функция, а значит, каждый раз новая компиляция.

In [ ]:
def unjitted_loop_body(prev_i):
    return prev_i + 1

def loop_with_lambda(n):
    i = 0
    for _ in range(n):
        i = jax.jit(lambda v: unjitted_loop_body(v))(i)     # плохо: новая lambda каждый раз
    return i

def loop_cached(n):
    i = 0
    body = jax.jit(unjitted_loop_body)                        # хорошо: одна функция
    for _ in range(n):
        i = body(i)
    return i

t0 = time.perf_counter(); loop_with_lambda(10); t_bad = time.perf_counter() - t0
t0 = time.perf_counter(); loop_cached(10); t_good = time.perf_counter() - t0
print(f"jit(lambda) в цикле: {t_bad*1e3:.0f} мс | один раз заджитили: {t_good*1e3:.0f} мс")

Если программа перекомпилируется слишком часто, включите диагностику:

In [ ]:
jax.config.update("jax_explain_cache_misses", True)

@jax.jit
def double(x):
    return 2 * x

_ = double(jnp.arange(3))      # промах кеша: функцию ещё не видели
_ = double(jnp.arange(3))      # попадание: тишина
_ = double(jnp.arange(3.0))    # промах: изменился dtype

jax.config.update("jax_explain_cache_misses", False)

> **Данные переменной формы** (например, последний неполный батч) — частый источник бесконечных перекомпиляций, ведь кеш зависит от формы. Стандартное решение: **дополнять** (pad) данные до небольшого набора фиксированных форм и маскировать лишнее. Другое простое решение — отбрасывать неполный батч (мы так и делаем в разделе 12).

## 6.6. `jit` и методы классов

Нельзя просто написать `@jax.jit` над методом: первый аргумент `self` не является типом JAX. Есть три стратегии.

1. **Вспомогательная jit-функция** — самая простая: `jit` только на отдельной функции с «чистыми» аргументами, а метод её вызывает.
2. **`self` как статический аргумент** — требует корректных `__hash__` и `__eq__` и **ломается, если объект изменяется** (кеш отдаёт устаревший код).
3. **Регистрация класса как pytree** — самая гибкая: массивы идут в `children`, статические настройки в `aux_data`.

In [ ]:
from jax import tree_util

class CustomClass:
    def __init__(self, x, mul: bool):
        self.x = x
        self.mul = mul

    @jax.jit
    def calc(self, y):
        if self.mul:               # mul — статическое поле, ветвиться можно
            return self.x * y
        return y

    def _tree_flatten(self):
        return (self.x,), {"mul": self.mul}

    @classmethod
    def _tree_unflatten(cls, aux_data, children):
        return cls(*children, **aux_data)

tree_util.register_pytree_node(CustomClass, CustomClass._tree_flatten, CustomClass._tree_unflatten)

c = CustomClass(2, True)
print(c.calc(3))              # 6
c.mul = False
print(c.calc(3))              # 3 — изменение поля учтено

## 6.7. Асинхронная отправка (asynchronous dispatch)

JAX **не ждёт** окончания вычисления, а сразу возвращает управление Python. Результат — «будущее» значение (future): оно будет получено на устройстве, но может быть ещё не готово. Python «бежит впереди» устройства, и накладные расходы Python не мешают.

Ждать приходится только когда вы смотрите на значение с хоста (печать, `np.asarray`, `float(...)`).

Из-за этого **наивный замер времени обманывает**:

In [ ]:
a = jax.random.uniform(jax.random.key(0), (3000, 3000))
jax.block_until_ready(a @ a)                                          # разогрев

t0 = time.perf_counter(); r = a @ a; t_dispatch = time.perf_counter() - t0
t0 = time.perf_counter(); r = a @ a; r.block_until_ready(); t_real = time.perf_counter() - t0

print(f"без block_until_ready: {t_dispatch*1e3:.2f} мс (только запуск)")
print(f"с block_until_ready:   {t_real*1e3:.2f} мс (реальное вычисление)")

## 6.8. Передача буферов (donation) — для любопытных

На границе `jit` выходные массивы получают **новые буферы памяти**, даже если вы просто обновляете параметры. Если вы уверены, что входной массив больше не нужен, можно **отдать** (donate) его буфер под результат:

```python
params, state = jax.jit(update_fn, donate_argnums=(0, 1))(params, state)
```

Это экономит память. После вызова отданный массив **нельзя** использовать (ошибка `Array has been deleted`). Подробнее — в документации. (На CPU передача буферов может не поддерживаться и просто выдаст предупреждение.)

## ✏️ Задание 6. jit-совместимый код


Реализуйте две функции так, чтобы они работали под `jax.jit` и `jax.vmap`.

1. `clip_by_norm(x, max_norm)` — если евклидова норма вектора `x` больше `max_norm`, масштабировать `x` так, чтобы норма стала равна `max_norm`; иначе вернуть `x` без изменений. **Без Python-`if`** (используйте `jnp.where` или `jnp.minimum`).
2. `collatz_steps(n)` — число шагов, за которое `n` (`int32`) превращается в 1 по правилу: чётное → `n // 2`, нечётное → `3 * n + 1`. Используйте `jax.lax.while_loop`. Для `n = 1` ответ `0`.

In [ ]:
def clip_by_norm(x, max_norm):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def collatz_steps(n):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def test_clip_by_norm_values():
    big = jnp.array([3.0, 4.0])      # норма 5
    small = jnp.array([0.3, 0.4])    # норма 0.5
    np.testing.assert_allclose(clip_by_norm(big, 1.0), [0.6, 0.8], rtol=1e-5, err_msg="длинный вектор нужно сжать до нормы 1")
    np.testing.assert_allclose(clip_by_norm(small, 1.0), [0.3, 0.4], rtol=1e-5, err_msg="короткий вектор менять не нужно")


def test_clip_by_norm_jit_and_vmap():
    xs = jnp.array([[3.0, 4.0], [0.3, 0.4], [0.0, 0.0]])
    out = jax.jit(jax.vmap(clip_by_norm, in_axes=(0, None)))(xs, 1.0)
    np.testing.assert_allclose(out, [[0.6, 0.8], [0.3, 0.4], [0.0, 0.0]], atol=1e-5)
    assert bool(jnp.all(jnp.isfinite(out))), "для нулевого вектора не должно быть nan"


def test_collatz_values():
    assert int(collatz_steps(jnp.int32(1))) == 0
    assert int(collatz_steps(jnp.int32(27))) == 111, "для 27 нужно 111 шагов"


def test_collatz_jit_vmap():
    ns = jnp.array([1, 2, 3, 6, 7], dtype=jnp.int32)
    out = jax.jit(jax.vmap(collatz_steps))(ns)
    np.testing.assert_array_equal(out, [0, 1, 7, 8, 16])


run_tests(test_clip_by_norm_values, test_clip_by_norm_jit_and_vmap, test_collatz_values, test_collatz_jit_vmap)

---
# 7. Компиляция заранее (Ahead-of-time, AOT)

Обычно `jit` работает **лениво**: трассировка и компиляция случаются при первом вызове. Но весь процесс можно разделить на явные **этапы** и выполнить их заранее:

```
Python-функция ──trace──▶ jaxpr ──lower──▶ StableHLO ──compile──▶ исполняемый файл
```

Зачем это нужно:

- **проверить** промежуточные представления (что именно скомпилировал JAX);
- получить **оценки** стоимости (FLOPs, память);
- **заранее** скомпилировать функцию (например, перед запуском сервиса), чтобы первый вызов не тормозил;
- скомпилировать функцию, имея только **форму и тип** входов, а не сами данные.

## 7.1. Этапы lower и compile

In [ ]:
def f(x, y):
    return jnp.sin(x) @ y

x = jnp.ones((4, 3), dtype=jnp.float32)
y = jnp.ones((3,), dtype=jnp.float32)

lowered = jax.jit(f).lower(x, y)             # этап 1: lowering -> StableHLO
print(lowered.as_text()[:700])

In [ ]:
compiled = lowered.compile()                  # этап 2: компиляция в исполняемый код
print("Результат:", compiled(x, y))

hlo = compiled.as_text()                      # оптимизированный HLO после работы XLA
print("\nПервые строки оптимизированной программы:\n", hlo[:500])

## 7.2. Оценка стоимости

У скомпилированной функции можно спросить оценку числа операций и объёма памяти (формат ответа зависит от версии JAX и бэкенда).

In [ ]:
cost = compiled.cost_analysis()
if isinstance(cost, (list, tuple)):
    cost = cost[0]
print("flops:", cost.get("flops"))
print("bytes accessed:", cost.get("bytes accessed"))

## 7.3. Компиляция без реальных данных: `ShapeDtypeStruct`

Вместо массивов можно передать **описание** формы и типа:

In [ ]:
spec_x = jax.ShapeDtypeStruct((4, 3), jnp.float32)
spec_y = jax.ShapeDtypeStruct((3,), jnp.float32)

compiled2 = jax.jit(f).lower(spec_x, spec_y).compile()
print(compiled2(x, y))

## 7.4. Скомпилированная функция «жёсткая»

Исполняемый файл рассчитан на **конкретные** формы и типы. Другая форма — ошибка (а не повторная трассировка, как у обычного `jit`).

In [ ]:
show_error(lambda: compiled(jnp.ones((5, 3), dtype=jnp.float32), y))
print("обычный jit справится с новой формой:", jax.jit(f)(jnp.ones((5, 3)), y).shape)

Статические аргументы при AOT тоже можно использовать: `jax.jit(f, static_argnums=...)` → `.lower(...)` → `.compile()`.

---
# 8. Управляющие конструкции и логические операторы с `jit`

Мы уже знаем: Python-`if` по **значению** внутри `jit`/`vmap` не работает. В JAX есть три группы инструментов.

| Что нужно | Что использовать |
|---|---|
| Ветвление по **статическому** значению (форма, флаг) | обычный Python `if`, при необходимости `static_argnums` |
| Поэлементный выбор | `jnp.where(cond, a, b)`, `jnp.select`, `jnp.piecewise` |
| Ветвление по значению, **одна** из двух веток | `lax.cond`, `lax.switch` |
| Циклы | `lax.fori_loop`, `lax.while_loop`, `lax.scan` |

## 8.1. Python-ветвление допустимо, если условие статическое

In [ ]:
@partial(jax.jit, static_argnames=["use_square"])
def maybe_square(x, use_square):
    if use_square:                 # статический флаг: нормально
        return x ** 2
    return x

print(maybe_square(3.0, True), maybe_square(3.0, False))

@jax.jit
def shape_dependent(x):
    if x.shape[0] > 2:             # форма известна при трассировке: тоже нормально
        return x.sum()
    return x.prod()

print(shape_dependent(jnp.arange(1.0, 5.0)), shape_dependent(jnp.arange(1.0, 3.0)))

## 8.2. `jnp.where`: оба результата считаются

`jnp.where(cond, a, b)` вычисляет **обе** ветки `a` и `b` и поэлементно выбирает. Это почти всегда нормально, но есть известная ловушка с градиентами.

In [ ]:
def safe_sqrt_bad(x):
    return jnp.where(x > 0, jnp.sqrt(x), 0.0)

print("значение  :", safe_sqrt_bad(-1.0))           # 0.0
print("градиент  :", jax.grad(safe_sqrt_bad)(-1.0)) # nan!

Почему `nan`? Ветвь `sqrt(x)` при $x=-1$ даёт `nan`, и при дифференцировании $0 \cdot \text{nan} = \text{nan}$. Лечится приёмом **«двойной where»**: сначала подменяем опасное значение безопасным, и только потом считаем.

In [ ]:
def safe_sqrt_good(x):
    safe_x = jnp.where(x > 0, x, 1.0)               # безопасный аргумент
    return jnp.where(x > 0, jnp.sqrt(safe_x), 0.0)

print("градиент  :", jax.grad(safe_sqrt_good)(-1.0))   # 0.0
print("градиент  :", jax.grad(safe_sqrt_good)(4.0))    # 0.25

## 8.3. `lax.cond` и `lax.switch`

`lax.cond(pred, true_fun, false_fun, *operands)` выполняет **одну** ветку по значению `pred`. Обе функции трассируются, и они должны возвращать результаты **одного типа и формы**.

In [ ]:
@jax.jit
def cond_example(x):
    return lax.cond(x > 0, lambda v: v * 2.0, lambda v: -v, x)

print(cond_example(3.0), cond_example(-3.0))
print("градиент через cond:", jax.grad(cond_example)(3.0), jax.grad(cond_example)(-3.0))

# switch: выбор из нескольких веток по индексу
branches = [lambda x: x + 1, lambda x: x * 10, lambda x: x ** 2]
print([float(lax.switch(i, branches, 3.0)) for i in range(3)])

> **Под `vmap` `cond` превращается в `select`**: обе ветки вычисляются для всего батча, потому что у разных элементов батча условие может быть разным.

## 8.4. Циклы: `fori_loop`, `while_loop`, `scan`

- `lax.fori_loop(lower, upper, body, init)` — цикл с фиксированным числом итераций.
- `lax.while_loop(cond, body, init)` — цикл «пока условие истинно».
- `lax.scan(f, init, xs)` — цикл по массиву с накоплением состояния, возвращает ещё и историю.

Состояние цикла (`carry`) должно сохранять **тип и форму** на каждой итерации.

In [ ]:
# fori_loop: сумма чисел от 0 до 9
print(lax.fori_loop(0, 10, lambda i, acc: acc + i, 0))

# while_loop: сколько раз можно удвоить 1, пока не станет >= 1000
print(lax.while_loop(lambda s: s[0] < 1000, lambda s: (s[0] * 2, s[1] + 1), (1, 0)))

# scan: числа Фибоначчи
def fib_step(carry, _):
    a, b = carry
    return (b, a + b), a

_, fibs = lax.scan(fib_step, (0, 1), None, length=10)
print(fibs)

**Почему не обычный цикл?** Python-цикл `for` под `jit` **разворачивается** при трассировке: каждая итерация попадает в программу отдельно. Для длинных циклов программа и время компиляции растут. `lax`-циклы записываются компактно.

In [ ]:
def python_loop(x):
    for _ in range(100):
        x = x * 1.01 + 0.1
    return x

def lax_loop(x):
    return lax.fori_loop(0, 100, lambda i, v: v * 1.01 + 0.1, x)

n_py = len(jax.make_jaxpr(python_loop)(1.0).eqns)
n_lax = len(jax.make_jaxpr(lax_loop)(1.0).eqns)
print("операций в jaxpr: python-цикл =", n_py, "| lax.fori_loop =", n_lax)

> **Дифференцирование.** `scan` и `fori_loop` с константными границами дифференцируются в обратном режиме. А вот `while_loop` с неизвестным числом итераций — **нет** (в обратном режиме) — потому что JAX должен сохранять промежуточные значения каждой итерации.

In [ ]:
def pow_while(x):
    return lax.while_loop(lambda s: s[1] < 3, lambda s: (s[0] * x, s[1] + 1), (1.0, 0))[0]

show_error(lambda: jax.grad(pow_while)(2.0))                 # reverse-mode: ошибка
print("прямой режим работает:", jax.jacfwd(pow_while)(2.0))   # d/dx x^3 = 12

## 8.5. Логические операторы

Операторы Python `and`, `or`, `not` требуют превратить значение в `bool`, а для трейсера это невозможно. Для массивов используйте **поэлементные** операторы:

| Python | JAX / NumPy |
|---|---|
| `a and b` | `a & b` или `jnp.logical_and(a, b)` |
| `a or b` | `a \| b` или `jnp.logical_or(a, b)` |
| `not a` | `~a` или `jnp.logical_not(a)` |
| «любой / все» | `jnp.any(a)`, `jnp.all(a)` |

> **Приоритет операторов!** `&` и `|` приоритетнее сравнений, поэтому **заключайте сравнения в скобки**: `(x > 0) & (x < 5)`. Без скобок получится `x > (0 & x) < 5`.

In [ ]:
x = jnp.arange(-2, 7)

show_error(lambda: jax.jit(lambda x: (x > 0) and (x < 5))(x))     # and: ошибка

in_range = jax.jit(lambda x: (x > 0) & (x < 5))(x)                 # & со скобками: ок
print(x)
print(in_range)
print("any:", jnp.any(in_range), "| all:", jnp.all(in_range))

---
# 9. Размещение данных (data placement)

Каждый массив JAX лежит **на устройстве**: CPU, GPU или TPU. Правило: **вычисление идёт там, где лежат данные**. Обычно JAX сам кладёт массивы на устройство по умолчанию (ускоритель, если он есть).

## 9.1. Какие устройства есть

In [ ]:
devices = jax.devices()
print("устройства:", devices)
print("backend:", jax.default_backend())
print("число локальных устройств:", jax.local_device_count())

x = jnp.arange(6.0)
print("x лежит на:", x.devices())

## 9.2. Явная пересылка: `device_put` и `device_get`

- `jax.device_put(x, device)` — отправить данные на устройство;
- `jax.device_get(x)` — получить данные обратно на хост (в NumPy).

Массивы NumPy живут на **хосте**. Если вы передаёте их в JAX-функцию в цикле, они **копируются на устройство при каждом вызове**. Если данные нужны много раз — один раз отправьте их на устройство.

In [ ]:
x_host = np.random.default_rng(0).normal(size=(2000, 2000)).astype(np.float32)   # данные на хосте
x_dev = jax.device_put(x_host)                                                    # копия на устройстве
print(type(x_host).__name__, "->", type(x_dev).__name__, "на", x_dev.devices())

@jax.jit
def heavy(a):
    return jnp.tanh(a @ a).sum()

print(f"из NumPy каждый раз: {bench(heavy, x_host):.2f} мс")
print(f"данные на устройстве: {bench(heavy, x_dev):.2f} мс")

back = jax.device_get(x_dev)
print("вернулись на хост:", type(back).__name__)

> На CPU разница небольшая (данные и так в памяти хоста). На GPU/TPU она может быть существенной.

## 9.3. Выбор устройства

Явно положить массив на нужное устройство можно через `device_put`. Устройство по умолчанию меняет контекст `jax.default_device`.

In [ ]:
cpu = jax.devices("cpu")[0]

x_cpu = jax.device_put(jnp.arange(4.0), cpu)
print("x_cpu на:", x_cpu.devices())

with jax.default_device(cpu):
    y = jnp.ones(3)
    print("создано внутри default_device:", y.devices())

## 9.4. Commited и uncommitted массивы

Массив, созданный без явного указания устройства, — «**не привязан**» (uncommitted): JAX может перемещать его по необходимости. Массив, который вы явно отправили на устройство (`device_put(x, dev)`), — «**привязан**» (committed): вычисление пойдёт там, а попытка смешать привязанные массивы с **разных** устройств даст ошибку.

In [ ]:
a = jnp.arange(3.0)
print("a привязан?", getattr(a, "committed", "n/a"))
b = jax.device_put(a, jax.devices()[0])
print("b привязан?", getattr(b, "committed", "n/a"))

devs = jax.devices()
if len(devs) > 1:
    p = jax.device_put(a, devs[0])
    q = jax.device_put(a, devs[1])
    show_error(lambda: p + q)         # привязаны к разным устройствам -> ошибка
else:
    print("Устройство одно — конфликт устройств показать не на чем (в Colab с TPU их 8).")

## 9.5. Один массив на нескольких устройствах: `sharding` (обзор)

Массив можно **разрезать** (shard) между устройствами. Описание задаётся объектом `sharding`: сетка устройств (`Mesh`) + правило разбиения (`PartitionSpec`). JAX потом сам распараллеливает вычисления над таким массивом. Это основа масштабирования на сотни устройств (подробности — в документации по sharding).

In [ ]:
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

mesh = Mesh(np.array(jax.devices()), ("data",))
sharding = NamedSharding(mesh, P("data"))          # разрезать по первой оси на устройствах оси 'data'

n = jax.device_count()
arr = jax.device_put(jnp.arange(8.0 * n), sharding)
print("sharding:", arr.sharding)
for s in arr.addressable_shards:
    print(" устройство", s.device, "хранит форму", s.data.shape)

print("сумма (посчитана на всех устройствах):", jnp.sum(arr))

---
# 10. Автодифференцирование: JVP и VJP

В разделе 2 мы пользовались `jax.grad`. Но он — лишь «парадный вход» в более глубокую систему, построенную на двух фундаментальных операциях: **JVP** (прямой режим) и **VJP** (обратный режим). Разберёмся, что они делают, на уровне математики.

## 10.1. Якобиан

Пусть $f:\mathbb{R}^n\to\mathbb{R}^m$ — дифференцируемая функция. Её **якобиан** в точке $x$ — матрица частных производных

$$
J_f(x)=\begin{pmatrix}
\dfrac{\partial f_1}{\partial x_1} & \cdots & \dfrac{\partial f_1}{\partial x_n}\\
\vdots & \ddots & \vdots\\
\dfrac{\partial f_m}{\partial x_1} & \cdots & \dfrac{\partial f_m}{\partial x_n}
\end{pmatrix}\in\mathbb{R}^{m\times n}.
$$

Он описывает **линейное приближение** функции: $f(x+\delta)\approx f(x)+J_f(x)\,\delta$.

Для скалярной функции ($m=1$) якобиан — это строка градиента: $J_f(x)=\nabla f(x)^\top$.

Полный якобиан часто слишком большой (у нейросети миллионы параметров и скалярный loss). К счастью, обычно нужны не его элементы, а **произведения** якобиана с вектором. Их можно считать дёшево, не строя матрицу целиком.

## 10.2. JVP: произведение якобиана на вектор (прямой режим)

**JVP** (*Jacobian-vector product*) при заданных $x\in\mathbb{R}^n$ и касательном векторе $v\in\mathbb{R}^n$ вычисляет

$$
\operatorname{jvp}(f)(x,v)=\bigl(f(x),\; J_f(x)\,v\bigr),\qquad J_f(x)\,v\in\mathbb{R}^m.
$$

Смысл: $J_f(x)\,v$ — это **производная $f$ вдоль направления $v$**, то есть $\dfrac{d}{dt}f(x+tv)\big|_{t=0}$.

Для составной функции $f=g\circ h$ цепное правило даёт $J_f=J_g(h(x))\,J_h(x)$. JVP идёт **слева направо** (по ходу вычисления): сначала $J_h v$, затем $J_g(J_h v)$. Касательный вектор «едет» вперёд вместе со значением.

Стоимость: примерно **как одно вычисление $f$** (в константу раз больше). Чтобы получить весь якобиан, нужно $n$ запусков JVP (по одному на базисный вектор $e_1,\dots,e_n$) — это даёт по одному **столбцу** за раз.

## 10.3. VJP: произведение вектора на якобиан (обратный режим)

**VJP** (*vector-Jacobian product*) при заданных $x$ и **котангенсе** $u\in\mathbb{R}^m$ вычисляет

$$
\operatorname{vjp}(f)(x,u)=\bigl(f(x),\; u^\top J_f(x)\bigr),\qquad u^\top J_f(x)\in\mathbb{R}^{1\times n}.
$$

Теперь вектор умножается **слева**: $u^\top J_g J_h = (u^\top J_g)\,J_h$. Вычисление идёт **справа налево** — от выхода к входу («обратное распространение»). В JAX функция `jax.vjp(f, x)` возвращает пару `(f(x), vjp_fn)`, где `vjp_fn(u)` считает $u^\top J_f(x)$.

Стоимость: тоже примерно «как одно вычисление $f$» (константа больше, чем у JVP), плюс **память** для хранения промежуточных значений прямого прохода. Чтобы получить весь якобиан, нужно $m$ запусков VJP (по одному на $e_1,\dots,e_m$) — это даёт по одной **строке** за раз.

## 10.4. Какой режим выбрать

| | JVP (`jax.jvp`, `jax.jacfwd`) | VJP (`jax.vjp`, `jax.grad`, `jax.jacrev`) |
|---|---|---|
| Умножает | $J\,v$ (справа) | $u^\top J$ (слева) |
| Направление | вперёд | назад |
| Весь якобиан за | $n$ проходов (**столбцы**) | $m$ проходов (**строки**) |
| Выгоден, когда | входов **мало** ($n\ll m$) | выходов **мало** ($m\ll n$) |
| Память | мало | хранит промежуточные значения |

**Градиент скалярной функции** ($m=1$): достаточно **одного** VJP с $u=1$:
$$\nabla f(x)^\top = 1\cdot J_f(x).$$
Именно поэтому `jax.grad` использует обратный режим: один проход обратного режима даёт градиент по **миллионам** параметров сразу. Для функции $\mathbb{R}\to\mathbb{R}^{1000}$ было бы наоборот — выгоднее JVP.

## 10.5. Практика: пример с якобианом

Возьмём $f:\mathbb{R}^2\to\mathbb{R}^3$,
$$
f(x_1,x_2)=\begin{pmatrix}x_1x_2\\ \sin x_1\\ x_2^2\end{pmatrix},\qquad
J_f(x)=\begin{pmatrix}x_2 & x_1\\ \cos x_1 & 0\\ 0 & 2x_2\end{pmatrix}.
$$

In [ ]:
def f(x):
    return jnp.array([x[0] * x[1], jnp.sin(x[0]), x[1] ** 2])

x = jnp.array([1.0, 2.0])

# якобиан, выписанный вручную
J_manual = jnp.array([[x[1], x[0]],
                      [jnp.cos(x[0]), 0.0],
                      [0.0, 2 * x[1]]])
print("J вручную:\n", J_manual)
print("jacfwd:\n", jax.jacfwd(f)(x))      # по столбцам (n=2 JVP)
print("jacrev:\n", jax.jacrev(f)(x))      # по строкам (m=3 VJP)

### JVP: `jax.jvp(f, (x,), (v,))` возвращает `(f(x), J v)`

In [ ]:
v = jnp.array([1.0, 0.5])
y, Jv = jax.jvp(f, (x,), (v,))
print("f(x)     =", y)
print("J v      =", Jv)
print("вручную  =", J_manual @ v)

### VJP: `jax.vjp(f, x)` возвращает `(f(x), vjp_fn)`

In [ ]:
y, vjp_fn = jax.vjp(f, x)
u = jnp.array([1.0, 0.0, 2.0])             # котангенс размерности m = 3
(uJ,) = vjp_fn(u)                           # возвращает кортеж — по одному элементу на каждый аргумент f
print("u^T J    =", uJ)
print("вручную  =", u @ J_manual)

### Якобиан по частям

Чтобы собрать весь якобиан из JVP, берём базисные векторы (столбцы); из VJP — строки. Так работают `jacfwd` и `jacrev`:

In [ ]:
n, m = 2, 3
cols = jnp.stack([jax.jvp(f, (x,), (e,))[1] for e in jnp.eye(n)], axis=1)     # столбцы через JVP
rows = jnp.stack([jax.vjp(f, x)[1](e)[0] for e in jnp.eye(m)], axis=0)         # строки через VJP
print("столбцы из JVP:\n", cols)
print("строки из VJP:\n", rows)

## 10.6. `grad` — это VJP с котангенсом 1

Для скалярной $f$ градиент — это VJP, к которому подан котангенс $u=1$.

In [ ]:
def scalar_f(x):
    return jnp.sum(jnp.sin(x) * x ** 2)

x = jnp.array([0.5, 1.0, 2.0])
val, vjp_fn = jax.vjp(scalar_f, x)
(g_vjp,) = vjp_fn(jnp.ones_like(val))
print("через vjp :", g_vjp)
print("jax.grad  :", jax.grad(scalar_f)(x))

## 10.7. Гессиан и произведение Гессиана на вектор

Гессиан — это якобиан градиента: $H_f(x)=J_{\nabla f}(x)\in\mathbb{R}^{n\times n}$. Для нейросети с $n=10^7$ параметров матрица $n\times n$ не поместится в память. Но **произведение $H v$** можно получить без самой матрицы:

$$
H_f(x)\,v=\operatorname{jvp}(\nabla f)(x,v)\ \Big(=\tfrac{d}{dt}\nabla f(x+tv)\Big|_{t=0}\Big).
$$

То есть берём JVP от функции `jax.grad(f)`. Это комбинация «**прямой режим поверх обратного**» (*forward-over-reverse*) — обычно самый эффективный способ.

In [ ]:
def hvp(f, x, v):
    return jax.jvp(jax.grad(f), (x,), (v,))[1]

v = jnp.array([1.0, 0.0, -1.0])
H = jax.hessian(scalar_f)(x)           # полный гессиан (3x3) — только для проверки
print("H v через hvp :", hvp(scalar_f, x, v))
print("H @ v         :", H @ v)

## 10.8. Собственные правила дифференцирования: `custom_vjp` (кратко)

Иногда автоматическая производная численно **неустойчива** или есть более эффективная формула. Тогда можно задать VJP вручную через `jax.custom_vjp`. Классический пример — функция $\operatorname{softplus}(x)=\log(1+e^x)$. Её производная равна $\sigma(x)=\dfrac{1}{1+e^{-x}}$. Наивная запись при больших $x$ приводит к `inf/inf = nan`.

In [ ]:
def softplus_naive(x):
    return jnp.log1p(jnp.exp(x))

print("наивный градиент при x=100:", jax.grad(softplus_naive)(100.0))     # nan

@jax.custom_vjp
def softplus(x):
    return jnp.logaddexp(0.0, x)                    # устойчивое вычисление значения

def softplus_fwd(x):
    return softplus(x), x                            # что сохранить для обратного прохода

def softplus_bwd(x, g):
    return (g * jax.nn.sigmoid(x),)                  # u^T J = g * sigma(x)

softplus.defvjp(softplus_fwd, softplus_bwd)

print("custom_vjp, значение  :", softplus(100.0))
print("custom_vjp, градиент  :", jax.grad(softplus)(100.0))                # 1.0
print("в нуле градиент       :", jax.grad(softplus)(0.0))                  # 0.5

## ✏️ Задание 7. JVP / VJP


1. `hvp(f, x, v)` — вернуть произведение гессиана скалярной функции `f` в точке `x` на вектор `v`, **не строя гессиан** (используйте `jax.jvp` от `jax.grad(f)`).
2. `grad_via_vjp(f, x)` — вернуть градиент скалярной `f` в точке `x`, но **только через `jax.vjp`** (не через `jax.grad`). Не забудьте про котангенс.

In [ ]:
def hvp(f, x, v):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def grad_via_vjp(f, x):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def _f_test(x):
    return jnp.sum(jnp.sin(x) * x ** 2) + jnp.prod(x)


def test_hvp_matches_hessian():
    x = jnp.array([0.5, 1.0, 2.0])
    v = jnp.array([1.0, -2.0, 0.5])
    expected = jax.hessian(_f_test)(x) @ v
    np.testing.assert_allclose(hvp(_f_test, x, v), expected, rtol=1e-4, atol=1e-5, err_msg="H v посчитано неверно")


def test_hvp_shape():
    x = jnp.arange(1.0, 6.0)
    assert hvp(_f_test, x, jnp.ones(5)).shape == (5,)


def test_grad_via_vjp():
    x = jnp.array([0.3, -1.2, 2.0])
    np.testing.assert_allclose(grad_via_vjp(_f_test, x), jax.grad(_f_test)(x), rtol=1e-5, atol=1e-6)


run_tests(test_hvp_matches_hessian, test_hvp_shape, test_grad_via_vjp)

---
# 11. JAX — The Sharp Bits (острые углы)

JAX очень мощный, но у него есть особенности, которые удивляют новичков. Ниже — основные из них. Многие мы уже видели. Здесь всё собрано в одном месте. Исходный материал: <https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html>

## 11.1. Чистые функции

JAX-преобразования рассчитаны на **функционально чистые** функции: результат зависит только от аргументов, побочных эффектов нет. Мы видели это в разделах 2, 5 и 6. Резюме:

- `print` и запись в глобальные переменные происходят при **трассировке**, а не при каждом вызове;
- чтение глобальной переменной «замораживает» её значение;
- изменяемое состояние нужно передавать **как аргументы и результаты**.

In [ ]:
g = 0.0

def impure_uses_global(x):
    return x + g

fast = jax.jit(impure_uses_global)
g = 0.0
print(fast(1.0))
g = 10.0
print(fast(1.0), "<- g изменили, а результат прежний (кеш)")

## 11.2. Нет изменения «на месте»

`x[i] = v` не работает — используйте `x.at[i].set(v)` (раздел 1.2).

## 11.3. Выход индекса за границы

Чтение обрезает индекс, запись отбрасывается (раздел 1.4). Для отрицательных индексов работает обычная логика Python.

In [ ]:
x = jnp.arange(5)
print(x[-1], x[100], x.at[100].set(0))
print("NaN вместо обрезания:", x.astype(float).at[100].get(mode="fill", fill_value=jnp.nan))

## 11.4. Входы — только массивы

`jnp.sum([1,2,3])` — ошибка (раздел 1.4). Так же и в функциях под `jit`: передавайте массивы или числа, а не списки.

## 11.5. Случайные числа

Нельзя использовать ключ повторно. Расщепляйте его (раздел 4).

## 11.6. Динамические формы под `jit`

Под `jit` **форма каждого массива должна быть известна** на этапе компиляции. Поэтому операции, у которых форма результата зависит от **значений** данных, не работают. Классический пример — булева маска.

In [ ]:
x = jnp.array([1.0, -2.0, 3.0, -4.0])

print("без jit:", x[x > 0])                                   # форма результата зависит от данных

show_error(lambda: jax.jit(lambda x: x[x > 0])(x))             # под jit — ошибка

Решения: выразить задачу через маску и `jnp.where` (результат фиксированной формы), либо передать **размер** явно (`size=` для `jnp.nonzero`, `jnp.unique`).

In [ ]:
@jax.jit
def sum_positive(x):
    return jnp.where(x > 0, x, 0.0).sum()           # форма не зависит от данных

@jax.jit
def positive_indices(x):
    return jnp.nonzero(x > 0, size=2, fill_value=-1)[0]   # размер задан явно

print(sum_positive(x), positive_indices(x))

## 11.7. Управляющие конструкции

Python-`if`/`while` по значению внутри `jit` не работают; используйте `lax.cond`, `lax.while_loop`, `lax.scan`, `jnp.where` (раздел 8). Помните и про:

- **двойной `where`** при градиентах (`nan` в «неиспользованной» ветке);
- **`and`/`or`/`not`** на массивах — вместо них `&`, `|`, `~` со скобками.

## 11.8. Точность: по умолчанию float32

JAX по умолчанию использует 32-битные числа. Для float64 включите флаг `jax_enable_x64` **в самом начале** программы (до создания массивов).

In [ ]:
a = jnp.array(1.0) / 3
print("float32:", a.dtype, f"{float(a):.17f}")

jax.config.update("jax_enable_x64", True)
b = jnp.array(1.0) / 3
print("float64:", b.dtype, f"{float(b):.17f}")
jax.config.update("jax_enable_x64", False)          # возвращаем как было

Целые тоже по умолчанию 32-битные, и переполнение **не выдаёт ошибку** — числа «заворачиваются»:

In [ ]:
big = jnp.array(2**31 - 1, dtype=jnp.int32)
print(big, "+ 1 =", big + 1)

## 11.9. Отладка NaN

Если в программе появляется `nan` и непонятно откуда, включите `jax_debug_nans`: JAX остановится на первой операции, создавшей `nan`, и покажет место. Режим замедляет программу, не оставляйте его включённым.

In [ ]:
jax.config.update("jax_debug_nans", True)
try:
    jax.jit(lambda x: jnp.log(x))(jnp.array(-1.0))
except Exception as e:
    print(type(e).__name__ + ":", str(e).split("\n")[0])
finally:
    jax.config.update("jax_debug_nans", False)

## 11.10. Печать внутри `jit`

Обычный `print` покажет трейсер. Для значений используйте `jax.debug.print` (и `jax.debug.breakpoint` для отладчика).

In [ ]:
@jax.jit
def f(x):
    print("print        :", x)
    jax.debug.print("debug.print  : {}", x)
    return x + 1

_ = f(jnp.array([1.0, 2.0]))

## 11.11. Асинхронность и замеры времени

Вычисления асинхронные: без `block_until_ready()` вы измеряете только запуск (раздел 6.7). Исключения могут «всплывать» позже, чем вызвана операция, их породившая.

## 11.12. Перекомпиляция

Новая форма или dtype → новая компиляция. Не вызывайте `jit` на новых lambda в цикле. Данные переменной длины паддят до фиксированных размеров (раздел 6.5). Для диагностики: `jax_explain_cache_misses`.

## 11.13. Память GPU

По умолчанию JAX **резервирует ~75%** памяти GPU при запуске, чтобы избежать фрагментации. Это нормально. Если рядом работает другой процесс, управляйте этим переменными окружения **до импорта JAX**:

```python
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"      # выделять по мере надобности
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".50"     # или ограничить долю
```

## 11.14. Точность матричных умножений на ускорителях

На GPU/TPU умножение матриц по умолчанию может выполняться с пониженной точностью (например, TF32 или bfloat16). Для повышения точности: аргумент `precision=` или `jax.default_matmul_precision`.

## 11.15. Работа с NumPy внутри `jit`

Внутри `jit` используйте `jnp`, а не `np`: функции NumPy не понимают трейсеры и сработают **на этапе трассировки**, либо упадут. NumPy внутри `jit` разумен только для статических значений (форм, констант).

In [ ]:
@jax.jit
def uses_numpy_bad(x):
    return np.sin(x)             # NumPy не умеет работать с трейсером

show_error(lambda: uses_numpy_bad(jnp.arange(3.0)))

### Краткий список правил

1. Чистые функции без побочных эффектов.
2. `x.at[i].set(v)` вместо `x[i] = v`.
3. Явные ключи случайности; ключ не используется дважды.
4. Нет ветвления по значению под `jit` — `jnp.where`, `lax.cond`, `lax.scan`.
5. Форма результата не должна зависеть от значений данных.
6. `jnp` вместо `np` внутри трассируемого кода.
7. `block_until_ready()` при измерении времени.
8. Не создавайте `jit` на лету в цикле.
9. Помните, что dtype по умолчанию — float32/int32.

---
# 12. Большой пример: логистическая регрессия (softmax) → нейросети

Соберём всё вместе: pytree параметров, `grad`, `jit`, случайные ключи, чистые функции. Решим задачу **бинарной классификации**, но запишем модель через **softmax** для $K=2$ классов. Это общая форма для многоклассовой классификации и для нейросетей: нужно будет заменить только функцию, которая считает логиты.

## 12.1. Математика

Данные: $(x_i,y_i)_{i=1}^N$, где $x_i\in\mathbb{R}^d$ — признаки, $y_i\in\{0,\dots,K-1\}$ — класс ($K=2$).

**Модель.** Параметры $W\in\mathbb{R}^{d\times K}$, $b\in\mathbb{R}^K$. **Логиты** (необработанные оценки классов):
$$
z = xW + b\in\mathbb{R}^K .
$$

**Softmax** превращает логиты в вероятности:
$$
p_k(x)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}},\qquad p_k\ge 0,\quad \sum_k p_k = 1 .
$$

**Функция потерь** — кросс-энтропия (отрицательное логарифмическое правдоподобие):
$$
\mathcal{L}(W,b)=-\frac1N\sum_{i=1}^{N}\log p_{y_i}(x_i).
$$

**Численная устойчивость.** Вместо $\log(\text{softmax}(z))$ считаем `log_softmax`: $\log p_k = z_k-\log\sum_j e^{z_j}$ (с приёмом `logsumexp`, вычитающим максимум). Так нет переполнения от $e^{z}$.

**Связь с сигмоидой.** При $K=2$:
$$
p_1=\frac{e^{z_1}}{e^{z_0}+e^{z_1}}=\frac{1}{1+e^{-(z_1-z_0)}}=\sigma(z_1-z_0).
$$
То есть softmax с двумя классами — это обычная логистическая регрессия, только с «лишним» параметром (важна только разность $z_1-z_0$).

**Градиент.** Пусть $P$ — матрица вероятностей $N\times K$, $Y$ — one-hot метки. Тогда
$$
\frac{\partial\mathcal{L}}{\partial z}=\frac{P-Y}{N},\qquad
\frac{\partial\mathcal{L}}{\partial W}=\frac1N X^\top(P-Y),\qquad
\frac{\partial\mathcal{L}}{\partial b}=\frac1N\sum_i (P-Y)_i .
$$
Мы сверим это с тем, что посчитает `jax.grad`.

**Обучение.** Мини-батчевый градиентный спуск: $\theta\leftarrow\theta-\eta\,\nabla_\theta\mathcal{L}_{\text{batch}}$, где $\theta=(W,b)$.

## 12.2. Данные

> 👉 **Ячейка ниже — место для ваших данных.** Замените её на свою: загрузите данные так, чтобы получились
> `X_train`, `X_test` (формы `(N, d)`, числа) и `y_train`, `y_test` (формы `(N,)`, метки `0` и `1`).
>
> Если переменные `X_train` и т.д. **уже определены**, ячейка ничего не меняет. Если нет — она создаёт искусственный набор данных, чтобы ноутбук работал «из коробки».

In [ ]:
# ============ ЗАГРУЗКА ДАННЫХ (замените эту ячейку своей) ============
# Нужно получить: X_train, X_test, y_train, y_test  (бинарная классификация, метки 0/1)

if "X_train" not in globals():
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split

    X_all, y_all = make_classification(
        n_samples=3000, n_features=10, n_informative=6, n_redundant=2,
        class_sep=1.0, flip_y=0.03, random_state=0)
    X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.25, random_state=0)
    print("Используются искусственные данные (sklearn.make_classification).")

print("X_train:", np.shape(X_train), "| X_test:", np.shape(X_test))
print("доля класса 1: train =", float(np.mean(y_train)), "| test =", float(np.mean(y_test)))

## 12.3. Подготовка данных

Данные загружают через NumPy/Pandas, а считают в JAX. Преобразуем их в `jnp`-массивы нужных типов (признаки — `float32`, метки — `int32`) и **стандартизируем** признаки. Среднее и стандартное отклонение берём **только из обучающей выборки**, чтобы не подглядывать в тест.

In [ ]:
Xtr_raw = jnp.asarray(np.asarray(X_train), dtype=jnp.float32)
Xte_raw = jnp.asarray(np.asarray(X_test), dtype=jnp.float32)
ytr = jnp.asarray(np.asarray(y_train), dtype=jnp.int32)
yte = jnp.asarray(np.asarray(y_test), dtype=jnp.int32)

mu = Xtr_raw.mean(axis=0)
sigma = Xtr_raw.std(axis=0) + 1e-8
Xtr = (Xtr_raw - mu) / sigma
Xte = (Xte_raw - mu) / sigma

n_features = Xtr.shape[1]
n_classes = 2
print("признаков:", n_features, "| классов:", n_classes)
print("среднее после стандартизации (train):", float(Xtr.mean()), "| std:", float(Xtr.std()))

## 12.4. Модель, функция потерь, точность

Параметры — **pytree** (словарь). Модель — чистая функция `logits_fn(params, X)`. Функции `make_loss` и `make_accuracy` принимают **любую** функцию логитов. Именно это даст нам «мост» к нейросетям.

In [ ]:
def init_logreg(key, n_features, n_classes=2):
    return {"W": 0.01 * jax.random.normal(key, (n_features, n_classes)),
            "b": jnp.zeros(n_classes)}


def logits_fn(params, X):
    # логиты z = X W + b, форма (N, K)
    return X @ params["W"] + params["b"]


def make_loss(forward):
    # возвращает функцию потерь (кросс-энтропия) для модели с логитами forward
    def loss(params, X, y):
        log_p = jax.nn.log_softmax(forward(params, X), axis=-1)             # (N, K)
        return -jnp.mean(jnp.take_along_axis(log_p, y[:, None], axis=1))     # -mean log p_y
    return loss


def make_accuracy(forward):
    def accuracy(params, X, y):
        return jnp.mean(jnp.argmax(forward(params, X), axis=-1) == y)
    return accuracy


loss_fn = make_loss(logits_fn)
accuracy_fn = make_accuracy(logits_fn)

params0 = init_logreg(jax.random.key(0), n_features)
print("форма W:", params0["W"].shape, "| форма b:", params0["b"].shape)
print("начальный loss:", float(loss_fn(params0, Xtr, ytr)), "(ожидаем ≈ ln 2 =", float(jnp.log(2.0)), ")")
print("начальная точность:", float(accuracy_fn(params0, Xtr, ytr)))

Начальный loss близок к $\ln 2\approx 0.693$: при почти нулевых весах модель выдаёт вероятности ≈ 0.5.

## 12.5. Проверка градиента

Сравним `jax.grad` с аналитическими формулами из 12.1 и проверим численно через `check_grads`.

In [ ]:
Xb, yb = Xtr[:256], ytr[:256]

g = jax.grad(loss_fn)(params0, Xb, yb)                      # pytree градиентов, та же структура, что params0

P = jax.nn.softmax(logits_fn(params0, Xb), axis=-1)
Y = jax.nn.one_hot(yb, n_classes)
dW_manual = Xb.T @ (P - Y) / Xb.shape[0]
db_manual = (P - Y).mean(axis=0)

print("dW совпадает с формулой:", bool(jnp.allclose(g["W"], dW_manual, atol=1e-6)))
print("db совпадает с формулой:", bool(jnp.allclose(g["b"], db_manual, atol=1e-6)))

check_grads(lambda p: loss_fn(p, Xb, yb), (params0,), order=1, modes=["rev"], atol=1e-2, rtol=1e-2)
print("check_grads пройден")

## 12.6. Обучение

Структура цикла обучения:

1. `make_train_step(loss)` строит **чистую**, скомпилированную (`jit`) функцию одного шага: `(params, X_батча, y_батча, lr) → (новые params, loss)`. Состояние (параметры) передаётся как аргумент и результат (раздел 5).
2. Шаг SGD — один `jax.tree.map` по параметрам и градиентам (раздел 3).
3. Случайность — через явные ключи `jax.random.split` (раздел 4). В каждой эпохе перемешиваем данные.
4. Все батчи одинакового размера (неполный хвост отбрасываем), чтобы `jit` не перекомпилировался (раздел 6.5).

In [ ]:
def make_train_step(loss):
    @jax.jit
    def train_step(params, Xb, yb, lr):
        l, grads = jax.value_and_grad(loss)(params, Xb, yb)
        params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
        return params, l
    return train_step


def fit(params, forward, X, y, *, X_val=None, y_val=None, epochs=30, batch_size=128, lr=0.1, key):
    loss = make_loss(forward)
    acc = make_accuracy(forward)
    train_step = make_train_step(loss)
    eval_loss, eval_acc = jax.jit(loss), jax.jit(acc)

    n_batches = X.shape[0] // batch_size
    hist = {"loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        key, sub = jax.random.split(key)
        perm = jax.random.permutation(sub, X.shape[0])
        for i in range(n_batches):
            idx = perm[i * batch_size:(i + 1) * batch_size]
            params, _ = train_step(params, X[idx], y[idx], lr)

        hist["loss"].append(float(eval_loss(params, X, y)))
        hist["train_acc"].append(float(eval_acc(params, X, y)))
        if X_val is not None:
            hist["val_acc"].append(float(eval_acc(params, X_val, y_val)))
    return params, hist

In [ ]:
t0 = time.perf_counter()
params_lr, hist_lr = fit(params0, logits_fn, Xtr, ytr, X_val=Xte, y_val=yte,
                         epochs=30, batch_size=128, lr=0.1, key=jax.random.key(1))
print(f"обучение заняло {time.perf_counter() - t0:.1f} с")

for ep in [0, 4, 9, 19, 29]:
    print(f"эпоха {ep+1:>2}: loss={hist_lr['loss'][ep]:.4f}  acc(train)={hist_lr['train_acc'][ep]:.3f}  acc(test)={hist_lr['val_acc'][ep]:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(hist_lr["loss"]); ax[0].set_title("Loss (train)"); ax[0].set_xlabel("эпоха")
ax[1].plot(hist_lr["train_acc"], label="train"); ax[1].plot(hist_lr["val_acc"], label="test")
ax[1].set_title("Точность"); ax[1].set_xlabel("эпоха"); ax[1].legend()
plt.tight_layout(); plt.show()

print("Итоговая точность на тесте:", float(accuracy_fn(params_lr, Xte, yte)))

## 12.7. Проверка: softmax при $K=2$ — это сигмоида

Вероятность класса 1 из softmax должна совпасть с $\sigma(z_1-z_0)$.

In [ ]:
z = logits_fn(params_lr, Xte)
p1_softmax = jax.nn.softmax(z, axis=-1)[:, 1]
p1_sigmoid = jax.nn.sigmoid(z[:, 1] - z[:, 0])
print("максимальная разница:", float(jnp.max(jnp.abs(p1_softmax - p1_sigmoid))))

# Предсказанные вероятности для первых 5 тестовых объектов
print(jnp.round(p1_softmax[:5], 3), "| истинные метки:", yte[:5])

## 12.8. Сравнение с `scikit-learn` (необязательно)

Для контроля сравним с готовой реализацией. Результаты должны быть близки. В sklearn по умолчанию включена L2-регуляризация, а у нас её нет, и мы учимся на мини-батчах. Поэтому разница в десятые доли процента — это нормально.

In [ ]:
try:
    from sklearn.linear_model import LogisticRegression
    sk = LogisticRegression(max_iter=1000).fit(np.asarray(Xtr), np.asarray(ytr))
    print("sklearn, точность на тесте:", sk.score(np.asarray(Xte), np.asarray(yte)))
    print("наша модель, точность на тесте:", float(accuracy_fn(params_lr, Xte, yte)))
except Exception as e:
    print("sklearn недоступен:", e)

## 12.9. Мост к нейросетям

Всё, что мы написали (`make_loss`, `make_accuracy`, `make_train_step`, `fit`), работает с **любой** функцией логитов. Чтобы получить нейросеть, достаточно изменить **только `forward` и инициализацию параметров**.

Нейросеть (MLP) — это та же формула $z=f_\theta(x)$, только $f_\theta$ теперь — композиция слоёв с нелинейностью:
$$
h_1=\operatorname{ReLU}(xW_1+b_1),\quad h_2=\operatorname{ReLU}(h_1W_2+b_2),\quad z=h_2W_3+b_3 .
$$
Слой softmax и кросс-энтропия — **те же**. Градиенты по всем $W_i,b_i$ считает тот же `jax.grad` (обратный режим, раздел 10), а параметры — тот же pytree (список словарей).

> Логистическая регрессия — это MLP **без скрытых слоёв**: `init_mlp(key, [d, 2])`.

In [ ]:
def init_mlp(key, sizes):
    keys = jax.random.split(key, len(sizes) - 1)
    return [{"W": jax.random.normal(k, (n_in, n_out)) * jnp.sqrt(2.0 / n_in), "b": jnp.zeros(n_out)}
            for k, n_in, n_out in zip(keys, sizes[:-1], sizes[1:])]


def mlp_logits(params, X):
    *hidden, last = params
    for layer in hidden:
        X = jax.nn.relu(X @ layer["W"] + layer["b"])
    return X @ last["W"] + last["b"]


mlp_params0 = init_mlp(jax.random.key(2), [n_features, 32, 32, n_classes])
print(jax.tree.map(lambda a: a.shape, mlp_params0))

mlp_params, hist_mlp = fit(mlp_params0, mlp_logits, Xtr, ytr, X_val=Xte, y_val=yte,
                           epochs=30, batch_size=128, lr=0.05, key=jax.random.key(3))

print("\nТочность на тесте:")
print("  логистическая регрессия:", round(hist_lr["val_acc"][-1], 4))
print("  MLP (2 скрытых слоя)   :", round(hist_mlp["val_acc"][-1], 4))

In [ ]:
plt.figure(figsize=(6, 3.6))
plt.plot(hist_lr["loss"], label="логистическая регрессия")
plt.plot(hist_mlp["loss"], label="MLP 32-32")
plt.xlabel("эпоха"); plt.ylabel("loss (train)"); plt.legend(); plt.title("Сравнение моделей")
plt.show()

Что мы получили:

- **модель** — чистая функция `logits_fn(params, X)`;
- **параметры** — pytree;
- **градиенты** — `jax.grad` по pytree;
- **шаг обучения** — `jit` + `tree.map`;
- **случайность** — явные ключи.

Так устроены и большие нейросети в JAX. Библиотеки Flax, Equinox, Optax лишь добавляют удобные слои и оптимизаторы поверх этих идей.

---
# 13. Итоговые задания

Эти задания используют модель и данные из раздела 12 (ячейки разделов 12.3–12.6 должны быть выполнены).

## ✏️ Задание 8. Кросс-энтропия, точность и L2-регуляризация


Реализуйте четыре функции. Нельзя использовать готовые `optax`/`flax`, но `jax.nn.log_softmax` использовать **нужно**.

1. `cross_entropy_from_logits(logits, y)` — средняя по батчу кросс-энтропия. `logits` имеет форму `(N, K)`, `y` — целые метки формы `(N,)`. Реализация должна быть **численно устойчивой**: для логитов порядка 1000 результат должен остаться конечным.
2. `accuracy_from_logits(logits, y)` — доля верных предсказаний (`argmax` по классам).
3. `l2_penalty(params)` — сумма квадратов элементов `params["W"]` (**смещение `b` не штрафуем**).
4. `loss_with_l2(params, X, y, lam)` — `cross_entropy_from_logits(logits_fn(params, X), y) + lam * l2_penalty(params)`.

Подсказка: `jnp.take_along_axis` или `logits[jnp.arange(N), y]` для выбора $\log p_{y_i}$.

In [ ]:
def cross_entropy_from_logits(logits, y):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def accuracy_from_logits(logits, y):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def l2_penalty(params):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def loss_with_l2(params, X, y, lam):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def test_cross_entropy_uniform():
    loss = cross_entropy_from_logits(jnp.zeros((3, 2)), jnp.array([0, 1, 0]))
    np.testing.assert_allclose(float(loss), np.log(2.0), rtol=1e-5, err_msg="при нулевых логитах loss = ln 2")


def test_cross_entropy_confident():
    loss = cross_entropy_from_logits(jnp.array([[10.0, 0.0], [0.0, 10.0]]), jnp.array([0, 1]))
    assert float(loss) < 1e-3, "уверенные верные предсказания должны давать почти нулевой loss"


def test_cross_entropy_stable():
    loss = cross_entropy_from_logits(jnp.array([[1000.0, 0.0]]), jnp.array([1]))
    assert bool(jnp.isfinite(loss)), "loss стал inf/nan — используйте log_softmax, а не log(softmax)"
    np.testing.assert_allclose(float(loss), 1000.0, rtol=1e-4)


def test_accuracy():
    logits = jnp.array([[2.0, 1.0], [0.0, 3.0], [5.0, 0.0], [1.0, 2.0]])
    y = jnp.array([0, 1, 1, 1])
    np.testing.assert_allclose(float(accuracy_from_logits(logits, y)), 0.75)


def test_l2_penalty_ignores_bias():
    p = {"W": jnp.array([[1.0, 2.0], [3.0, 4.0]]), "b": jnp.array([10.0, 10.0])}
    np.testing.assert_allclose(float(l2_penalty(p)), 30.0, err_msg="сумма квадратов W = 1+4+9+16 = 30, b не входит")


def test_loss_with_l2_gradient_matches_analytic():
    k1, k2, k3 = jax.random.split(jax.random.key(0), 3)
    X = jax.random.normal(k1, (20, 4))
    y = jax.random.randint(k2, (20,), 0, 2)
    params = {"W": jax.random.normal(k3, (4, 2)), "b": jnp.array([0.1, -0.2])}
    lam = 0.3
    g = jax.grad(loss_with_l2)(params, X, y, lam)
    P = jax.nn.softmax(X @ params["W"] + params["b"], axis=-1)
    Y = jax.nn.one_hot(y, 2)
    dW = X.T @ (P - Y) / 20 + 2 * lam * params["W"]
    db = (P - Y).mean(axis=0)
    np.testing.assert_allclose(g["W"], dW, rtol=1e-4, atol=1e-5, err_msg="dL/dW не совпал с аналитической формулой")
    np.testing.assert_allclose(g["b"], db, rtol=1e-4, atol=1e-5, err_msg="dL/db не совпал (b не должно штрафоваться)")


run_tests(test_cross_entropy_uniform, test_cross_entropy_confident, test_cross_entropy_stable,
          test_accuracy, test_l2_penalty_ignores_bias, test_loss_with_l2_gradient_matches_analytic)

### Применим ваше решение к обучению

После успешных тестов запустите ячейку ниже. Она обучает модель с L2-регуляризацией на ваших данных и сравнивает точность с моделью без регуляризации. (Если задание 8 ещё не решено, ячейка сообщит об этом.)

In [ ]:
def fit_l2(params, X, y, lam, *, epochs=30, batch_size=128, lr=0.1, key):
    @jax.jit
    def step(params, Xb, yb):
        g = jax.grad(loss_with_l2)(params, Xb, yb, lam)
        return jax.tree.map(lambda p, gr: p - lr * gr, params, g)

    for _ in range(epochs):
        key, sub = jax.random.split(key)
        perm = jax.random.permutation(sub, X.shape[0])
        for i in range(X.shape[0] // batch_size):
            idx = perm[i * batch_size:(i + 1) * batch_size]
            params = step(params, X[idx], y[idx])
    return params

try:
    for lam in [0.0, 0.01, 0.1]:
        p = fit_l2(params0, Xtr, ytr, lam, key=jax.random.key(1))
        acc_te = float(accuracy_from_logits(logits_fn(p, Xte), yte))
        print(f"lam={lam:<5} | ||W||^2={float(l2_penalty(p)):8.3f} | точность на тесте={acc_te:.4f}")
except NotImplementedError:
    print("⏳ Сначала выполните задание 8.")

## ✏️ Задание 9. SGD с импульсом на pytree


Реализуйте шаг градиентного спуска **с импульсом** (momentum) для произвольных pytree параметров:

$$v_{new}=\beta\,v+g,\qquad p_{new}=p-\eta\,v_{new}.$$

Функция `momentum_step(params, velocity, grads, lr, beta)` возвращает пару `(new_params, new_velocity)`. Все три входных pytree имеют **одинаковую структуру**. Используйте `jax.tree.map`. Функция должна быть чистой (ничего не менять на месте) и работать под `jit`.

In [ ]:
def momentum_step(params, velocity, grads, lr, beta):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def test_momentum_step_numbers():
    params = {"w": jnp.array([1.0, 2.0]), "b": jnp.array(0.5)}
    velocity = {"w": jnp.array([0.1, 0.1]), "b": jnp.array(0.0)}
    grads = {"w": jnp.array([1.0, -1.0]), "b": jnp.array(2.0)}
    new_p, new_v = momentum_step(params, velocity, grads, lr=0.1, beta=0.9)
    np.testing.assert_allclose(new_v["w"], [0.09 + 1.0, 0.09 - 1.0], rtol=1e-5, err_msg="v_new = beta*v + g")
    np.testing.assert_allclose(new_p["w"], [1.0 - 0.1 * 1.09, 2.0 + 0.1 * 0.91], rtol=1e-5, err_msg="p_new = p - lr*v_new")
    np.testing.assert_allclose(new_p["b"], 0.5 - 0.1 * 2.0, rtol=1e-5)


def test_momentum_step_structure_and_purity():
    params = [{"w": jnp.ones(2)}, {"w": jnp.ones(3)}]
    velocity = jax.tree.map(jnp.zeros_like, params)
    grads = jax.tree.map(jnp.ones_like, params)
    new_p, new_v = momentum_step(params, velocity, grads, 0.1, 0.9)
    assert jax.tree.structure(new_p) == jax.tree.structure(params), "структура params должна сохраниться"
    assert jax.tree.structure(new_v) == jax.tree.structure(params), "структура velocity должна сохраниться"
    assert bool(jnp.all(params[0]["w"] == 1.0)) and bool(jnp.all(velocity[0]["w"] == 0.0)), "входы менять нельзя"


def test_momentum_minimizes_quadratic():
    loss = lambda p: jnp.sum(p["w"] ** 2)
    params = {"w": jnp.array([3.0, -2.0, 1.0])}
    velocity = jax.tree.map(jnp.zeros_like, params)

    @jax.jit
    def step(params, velocity):
        return momentum_step(params, velocity, jax.grad(loss)(params), 0.05, 0.9)

    for _ in range(300):
        params, velocity = step(params, velocity)
    assert float(loss(params)) < 1e-4, f"loss не уменьшился достаточно: {float(loss(params)):.2e}"


run_tests(test_momentum_step_numbers, test_momentum_step_structure_and_purity, test_momentum_minimizes_quadratic)

### Дополнительно (без тестов)

1. Постройте график зависимости точности на тесте от `lam` для L2-регуляризации (задание 8).
2. Используйте `momentum_step` из задания 9 вместо обычного SGD в функции `fit` и сравните скорость сходимости.
3. Замените ReLU в `mlp_logits` на `jnp.tanh`. Что изменилось?
4. Посчитайте **градиенты по каждому примеру** (`vmap` + `grad`, раздел 2) для модели из раздела 12 и посмотрите на распределение норм этих градиентов.
5. Сделайте функцию `fit` полностью внутри `jit`: цикл по батчам через `lax.scan`, перемешивание — через `jax.random.permutation` внутри функции.

## Контрольные вопросы

1. Почему `x[0] = 1` не работает в JAX и чем его заменить?
2. Почему `print` внутри `jit`-функции срабатывает только один раз? Как печатать значения при каждом запуске?
3. Чем отличаются JVP и VJP? Почему `jax.grad` использует обратный режим?
4. Почему ключ случайности нельзя использовать дважды?
5. Почему `jnp.where(x > 0, jnp.sqrt(x), 0.)` может дать `nan` в градиенте?
6. Когда Python-`if` допустим внутри `jit`, а когда нужен `lax.cond`?
7. Что произойдёт, если вызвать `jit`-функцию с массивом другой формы?
8. Что нужно изменить в коде логистической регрессии, чтобы получить нейросеть?

## Что читать дальше

- Документация JAX: <https://docs.jax.dev> (разделы 101, 201, 301)
- Sharp Bits: <https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html>
- Библиотеки: **Optax** (оптимизаторы), **Flax** / **Equinox** (нейросети).

---
# Эталонные решения

⚠️ Сначала попробуйте решить задания сами. Нажмите на название задания, чтобы раскрыть решение.

<details>
<summary><b>Задание 1. Массивы и неизменяемость</b></summary>

```python
def zero_first_row(x):
    return x.at[0].set(0)


def standardize_columns(x):
    return (x - x.mean(axis=0)) / x.std(axis=0)
```

</details>

<details>
<summary><b>Задание 2. grad и vmap</b></summary>

```python
def f2(x):
    return jnp.sum(x ** 3) + jnp.sin(x[0])


def grad_f2(x):
    return jax.grad(f2)(x)


def batch_grad_f2(xs):
    return jax.vmap(jax.grad(f2))(xs)
```

</details>

<details>
<summary><b>Задание 3. Pytree</b></summary>

```python
def tree_scale(tree, c):
    return jax.tree.map(lambda x: x * c, tree)


def global_norm(tree):
    leaves = jax.tree.leaves(tree)
    return jnp.sqrt(sum(jnp.sum(x ** 2) for x in leaves))
```

</details>

<details>
<summary><b>Задание 4. Случайные числа</b></summary>

```python
def two_noises(key, n):
    k1, k2 = jax.random.split(key)
    return jax.random.normal(k1, (n,)), jax.random.normal(k2, (n,))
```

</details>

<details>
<summary><b>Задание 5. Чистая функция с состоянием</b></summary>

```python
def update_mean(state, x):
    count, mean = state
    count = count + 1
    mean = mean + (x - mean) / count
    return (count, mean), mean


def running_means(xs):
    init = (jnp.float32(0), jnp.float32(0))
    _, means = jax.lax.scan(update_mean, init, xs)
    return means
```

</details>

<details>
<summary><b>Задание 6. jit-совместимый код</b></summary>

```python
def clip_by_norm(x, max_norm):
    norm = jnp.linalg.norm(x)
    return x * jnp.minimum(1.0, max_norm / (norm + 1e-12))


def collatz_steps(n):
    def cond(state):
        m, _ = state
        return m != 1

    def body(state):
        m, k = state
        m = jnp.where(m % 2 == 0, m // 2, 3 * m + 1)
        return m, k + 1

    _, steps = jax.lax.while_loop(cond, body, (n, jnp.zeros_like(n)))
    return steps
```

</details>

<details>
<summary><b>Задание 7. JVP / VJP</b></summary>

```python
def hvp(f, x, v):
    return jax.jvp(jax.grad(f), (x,), (v,))[1]


def grad_via_vjp(f, x):
    y, vjp_fn = jax.vjp(f, x)
    return vjp_fn(jnp.ones_like(y))[0]
```

</details>

<details>
<summary><b>Задание 8. Кросс-энтропия, точность и L2-регуляризация</b></summary>

```python
def cross_entropy_from_logits(logits, y):
    log_p = jax.nn.log_softmax(logits, axis=-1)
    return -jnp.mean(log_p[jnp.arange(logits.shape[0]), y])


def accuracy_from_logits(logits, y):
    return jnp.mean(jnp.argmax(logits, axis=-1) == y)


def l2_penalty(params):
    return jnp.sum(params["W"] ** 2)


def loss_with_l2(params, X, y, lam):
    return cross_entropy_from_logits(logits_fn(params, X), y) + lam * l2_penalty(params)
```

</details>

<details>
<summary><b>Задание 9. SGD с импульсом на pytree</b></summary>

```python
def momentum_step(params, velocity, grads, lr, beta):
    new_velocity = jax.tree.map(lambda v, g: beta * v + g, velocity, grads)
    new_params = jax.tree.map(lambda p, v: p - lr * v, params, new_velocity)
    return new_params, new_velocity
```

</details>